# Notebook 02 — Data Preparation

I prepare one dataset at a time and leave the original source files alone. Every dataset goes through the same four steps: pick my 660 areas, look at every field, keep the measures I want, then save and check. All of them are keyed on the MSOA area code, so they can be joined later. I haven't joined anything yet. This notebook only gets the data ready.

First I check the environment: the Python version, whether I'm inside the virtual environment, and which library versions I have.

In [1]:
import sys
from importlib.metadata import version
import pandas as pd

print("Python version:", sys.version.split()[0])
print("Running inside a virtual environment:", sys.prefix != sys.base_prefix)
print("Environment folder name:", __import__("pathlib").Path(sys.prefix).name)

packages = ["pandas", "openpyxl", "geopandas", "matplotlib", "ipykernel"]
pd.DataFrame({
    "Library": packages,
    "Installed version": [version(package) for package in packages],
})

Python version: 3.14.7
Running inside a virtual environment: True
Environment folder name: .venv


,Library,Installed version
0,pandas,3.0.6
1,openpyxl,3.1.5
2,geopandas,1.2.0
3,matplotlib,3.11.2
4,ipykernel,7.4.0


Python 3.14.7, running inside `.venv`, with the five libraries listed. That's what I expected. If it ever shows something different, I check which kernel I've selected before going on.

## 1. Healthy life expectancy (HLE)

HLE is the number of years a person is expected to spend in good health. It's my main dataset. I put it on the map first, look at where it varies, and then ask what else about an area might explain the pattern. I chose it over plain life expectancy because it adds how healthy those years are. [Notebook 01](../A%20%E2%80%94%20Source%20Exploration/01_Source_Exploration.ipynb) explores it in detail, and here I just prepare it.

The ONS workbook has several sheets. Sheet `2` is healthy life expectancy at birth for 2019–2023, with separate male and female rows. Sheet `1` is life expectancy and sheet `3` is disability-free life expectancy, so it matters that I pick the right one. `header=6` tells pandas the column names are on the seventh row of the sheet.

In [2]:
from pathlib import Path
from IPython.display import display

# Find the project from the notebook's current folder.
working_folder = Path.cwd()
project_folder = next(
    folder for folder in [working_folder, *working_folder.parents]
    if (folder / "Data/Raw").is_dir() or (folder / "data/raw").is_dir()
)
data_folder = project_folder / ("Data" if (project_folder / "Data").is_dir() else "data")

def project_data(relative_path):
    """Locate a data file or folder in either project layout."""
    parts = Path(relative_path).parts
    if data_folder.name == "data":
        parts = (parts[0].lower(), *parts[1:])
    return data_folder.joinpath(*parts)

def raw_file(filename):
    """Find an untouched source in the project's raw collection."""
    candidates = sorted(project_data("Raw").rglob(filename))
    if not candidates:
        # The working collection also contains the original Unit 2 downloads.
        candidates = sorted((project_folder.parent.parent / "Datasets/Raw").rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"Raw source not found: {filename}")
    return candidates[0]

hle_workbook = raw_file("ons_hsle_msoa_2019_2023.xlsx")
# Sheet 2 is HLE; its column headings start on worksheet row 7.
hle_raw = pd.read_excel(hle_workbook, sheet_name="2", header=6)

print(f"Source: {len(hle_raw):,} rows; {len(hle_raw.columns)} fields")
display(pd.DataFrame({"Source field": hle_raw.columns}))
display(hle_raw[["Area code", "Area name", "Sex", "HLE"]].head())

Source: 15,355 rows; 13 fields


,Source field
0,Period
1,Country
2,Area type
3,Area code
4,Area name
5,Sex
6,Sex code
7,HLE
8,LCI
9,UCI


,Area code,Area name,Sex,HLE
0,E02000001,City of London 001,Male,72.4
1,E02000001,City of London 001,Female,74.6
2,E02000002,Barking and Dagenham 001,Male,60.2
3,E02000002,Barking and Dagenham 001,Female,60.1
4,E02000003,Barking and Dagenham 002,Male,65.0


### Which fields I keep

One row is one area, one sex and one period. I keep male and female apart, and I use the area code as the link to the map and to every other dataset.

The compact table has four fields: area code, local area name, sex and HLE in years. The period is 2019–2023 for every row, so I say it here and don't repeat it in a column. The confidence intervals, the proportion of life in good health and the two imputation flags go into a supporting table. I don't drop flagged estimates, and I don't turn missing values into zeros.

I start by picking the 660 areas in my boundary file. That's the coverage I have, not a final study boundary.

In [3]:
import geopandas as gpd

boundary_file = raw_file("ons_msoa21_boundaries_exploratory_west_midlands.geojson")
area_boundaries = gpd.read_file(boundary_file)
coverage_codes = set(area_boundaries["MSOA21CD"])
assert area_boundaries["MSOA21CD"].is_unique, "Repeated boundary codes"

# Match statistical area codes, keeping both sexes within the available map coverage.
hle_coverage = hle_raw.loc[
    hle_raw["Area code"].isin(coverage_codes)
    & hle_raw["Area type"].eq("MSOA")
    & hle_raw["Period"].eq("2019 to 2023")
].copy()
missing_hle_codes = coverage_codes - set(hle_coverage["Area code"])
assert not missing_hle_codes, f"Coverage areas without HLE rows: {missing_hle_codes}"
# One row per area and sex is expected for this period.
assert not hle_coverage.duplicated(["Area code", "Sex", "Period"]).any()
assert hle_coverage["Country"].eq("England").all()
assert set(hle_coverage["Sex"]) == {"Male", "Female"}
assert hle_coverage.groupby("Area code")["Sex"].nunique().eq(2).all()

print(f"Available areas: {hle_coverage['Area code'].nunique():,}")
print(f"HLE observations: {len(hle_coverage):,} — male and female rows kept separately")
print(f"Coverage areas without HLE rows: {len(missing_hle_codes)}")

Available areas: 660
HLE observations: 1,320 — male and female rows kept separately
Coverage areas without HLE rows: 0


### The compact table

`hle_support` keeps every source field for my areas, and `hle_display` keeps the four I need. HLE stays exactly as published: I don't round, recalculate or fill anything. The area names are still the ONS ones at this point. I swap them next.

In [4]:
# Keep the source evidence separately before narrowing the displayed table.
hle_support = hle_coverage.copy()
hle_display = hle_support[
    ["Area code", "Area name", "Sex", "HLE"]
].rename(columns={
    "Area code": "area_code", "Area name": "area_name",
    "Sex": "sex", "HLE": "hle_years",
}).copy()

assert len(hle_display) == len(hle_support)
assert not hle_display.duplicated(["area_code", "sex"]).any()
assert pd.api.types.is_numeric_dtype(hle_display["hle_years"])
print(f"Fields: {len(hle_support.columns)} supporting → {len(hle_display.columns)} compact")
print(f"Missing HLE estimates: {hle_display['hle_years'].isna().sum()}")
display(hle_display.head(10))
display(hle_support[["LE imputed", "Health prevalence imputed"]].value_counts(dropna=False).rename("Rows").reset_index())

Fields: 13 supporting → 4 compact
Missing HLE estimates: 0


,area_code,area_name,sex,hle_years
3492,E02001827,Birmingham 001,Male,68.4
3493,E02001827,Birmingham 001,Female,68.3
3494,E02001828,Birmingham 002,Male,69.5
3495,E02001828,Birmingham 002,Female,72.6
3496,E02001829,Birmingham 003,Male,73.3
3497,E02001829,Birmingham 003,Female,73.3
3498,E02001830,Birmingham 004,Male,71.7
3499,E02001830,Birmingham 004,Female,72.4
3500,E02001831,Birmingham 005,Male,63.7
3501,E02001831,Birmingham 005,Female,63.9


,LE imputed,Health prevalence imputed,Rows
0,0,0,1311
1,1,0,8
2,2,0,1


### Local area names

ONS calls its areas Birmingham 001, Birmingham 002 and so on. That's fine for a statistics office, but no use if you want to spot Sparkbrook or Tyseley on a map. My audience knows the city, so I swap the numbers for familiar names.

The names come from the House of Commons Library MSOA names, adapted for my Concept B prototype and saved in `Data/Reference/Area Names/`. The lookup covers all 660 areas. Some names carry a qualifier such as East or West, so Acocks Green East and Acocks Green West stay separate. I match by area code, so a new name never changes a boundary or an HLE value. The attribution and licence are kept in `naming-source.json` and `msoa-names-source.csv`.

The labels are provisional, because one MSOA can cover several neighbourhoods. I'll review them against the boundaries.

In [5]:
import json

local_names_file = project_data("Reference/Area Names/locality-names.json")
with local_names_file.open(encoding="utf-8") as names_file:
    local_names = json.load(names_file)

# Keep useful qualifiers: Acocks Green East and West must remain distinguishable.
local_name_by_code = {
    area_code: " ".join(
        part.strip() for part in [entry["display_name"], entry.get("qualifier", "")]
        if part.strip()
    )
    for area_code, entry in local_names.items()
}
missing_name_codes = set(hle_display["area_code"]) - set(local_name_by_code)
assert not missing_name_codes, f"Area codes without local names: {sorted(missing_name_codes)}"

# Match by code so the familiar name stays attached to the correct HLE estimate.
new_area_names = hle_display["area_code"].map(local_name_by_code)
assert new_area_names.notna().all() and new_area_names.str.strip().ne("").all()
# Check that replacing names changes no other values.
hle_before_renaming = hle_display.drop(columns="area_name").copy()
hle_display["area_name"] = new_area_names
pd.testing.assert_frame_equal(hle_display.drop(columns="area_name"), hle_before_renaming)

display(hle_display.head(10).rename(columns={"area_name": "local_area_name"}))
print(f"Local names applied to {hle_display['area_code'].nunique():,} areas; {len(hle_display):,} HLE rows.")
print("Area codes and HLE values unchanged.")

,area_code,local_area_name,sex,hle_years
3492,E02001827,Hill Hook,Male,68.4
3493,E02001827,Hill Hook,Female,68.3
3494,E02001828,Four Oaks,Male,69.5
3495,E02001828,Four Oaks,Female,72.6
3496,E02001829,Little Sutton,Male,73.3
3497,E02001829,Little Sutton,Female,73.3
3498,E02001830,Sutton Coldfield North,Male,71.7
3499,E02001830,Sutton Coldfield North,Female,72.4
3500,E02001831,Reddicap Heath,Male,63.7
3501,E02001831,Reddicap Heath,Female,63.9


Local names applied to 660 areas; 1,320 HLE rows.
Area codes and HLE values unchanged.


### Save the cleaned HLE table

I save it as `Data/Cleaned/HLE/hle_2019_2023_msoa.csv`. `Raw` holds the untouched source and `Cleaned` holds everything I've prepared. On a rerun, the code keeps a timestamped copy of the old file first. Then it reads the saved file back and checks the columns, row count and values.

In [6]:
from datetime import datetime
import shutil

clean_hle = hle_display.rename(columns={"area_name": "local_area_name"}).copy()
clean_hle = clean_hle[["area_code", "local_area_name", "sex", "hle_years"]]
assert not clean_hle.duplicated(["area_code", "sex"]).any()
assert clean_hle["local_area_name"].notna().all()

clean_folder = project_data("Cleaned/HLE")
clean_folder.mkdir(parents=True, exist_ok=True)
clean_file = clean_folder / "hle_2019_2023_msoa.csv"

# Retain the previous saved dataset before replacing it on a rerun.
if clean_file.exists():
    backup_folder = project_folder / "VS_CODE/Backups/Outputs/HLE"
    backup_folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    shutil.copy2(clean_file, backup_folder / f"hle_2019_2023_msoa - {stamp}.csv")

clean_hle.to_csv(clean_file, index=False)
# Read the file back to check what actually reached disk.
saved_hle = pd.read_csv(clean_file)
pd.testing.assert_frame_equal(saved_hle, clean_hle.reset_index(drop=True))
print("Saved: Data/Cleaned/HLE/hle_2019_2023_msoa.csv")
print(f"Verified: {len(saved_hle):,} rows, {saved_hle['area_code'].nunique():,} areas, {len(saved_hle.columns)} columns.")
display(saved_hle.head(10))

Saved: Data/Cleaned/HLE/hle_2019_2023_msoa.csv
Verified: 1,320 rows, 660 areas, 4 columns.


,area_code,local_area_name,sex,hle_years
0,E02001827,Hill Hook,Male,68.4
1,E02001827,Hill Hook,Female,68.3
2,E02001828,Four Oaks,Male,69.5
3,E02001828,Four Oaks,Female,72.6
4,E02001829,Little Sutton,Male,73.3
5,E02001829,Little Sutton,Female,73.3
6,E02001830,Sutton Coldfield North,Male,71.7
7,E02001830,Sutton Coldfield North,Female,72.4
8,E02001831,Reddicap Heath,Male,63.7
9,E02001831,Reddicap Heath,Female,63.9


HLE is saved: 1,320 rows for 660 areas, two per area. Values run from 52.4 years (Bloxwich South, male) to 76.6 (Kingswood, female).

## 2. Household deprivation

I do the data I already have before I go looking for more. First is Census 2021 table TS011, household deprivation. A household counts as deprived in a dimension if it meets a condition on education, employment, health and disability, or housing. The source gives the number of households deprived in none, one, two, three or four dimensions. It isn't a measure of debt or income.

I want to compare the share of households facing several kinds of disadvantage with the HLE pattern. The health and disability dimension overlaps with what HLE measures, so any link will partly reflect that. I read the MSOA file straight from the source ZIP. The source gives counts, and I need percentages to compare areas with different numbers of households.

In [7]:
from zipfile import ZipFile

# This collected source is in the Unit 2 raw collection, above the project folder.
deprivation_zip = raw_file("census2021-ts011.zip")
with ZipFile(deprivation_zip) as archive:
    with archive.open("census2021-ts011-msoa.csv") as source_file:
        deprivation_raw = pd.read_csv(source_file)
print(f"Source: {len(deprivation_raw):,} MSOA rows; {len(deprivation_raw.columns)} fields")
display(pd.DataFrame({"Source field": deprivation_raw.columns}))
display(deprivation_raw.loc[deprivation_raw["geography"].str.startswith("Birmingham", na=False)].iloc[:5, :4])

Source: 7,264 MSOA rows; 9 fields


,Source field
0,date
1,geography
2,geography code
3,Household deprivation: Total: All households; ...
4,Household deprivation: Household is not depriv...
5,Household deprivation: Household is deprived i...
6,Household deprivation: Household is deprived i...
7,Household deprivation: Household is deprived i...
8,Household deprivation: Household is deprived i...


,date,geography,geography code,Household deprivation: Total: All households; measures: Value
2922,2021,Birmingham 001,E02001827,2634
2923,2021,Birmingham 002,E02001828,3130
2924,2021,Birmingham 003,E02001829,3569
2925,2021,Birmingham 004,E02001830,3988
2926,2021,Birmingham 005,E02001831,4493


### Pick my areas and look at every field

I pick the same 660 area codes as HLE. That filters the places and keeps all nine columns. The sideways sample shows five Birmingham areas in source order, not the most deprived, so the long Census headings read down the page.

In [8]:
assert deprivation_raw["geography code"].is_unique, "Repeated source area codes"
assert deprivation_raw["date"].astype(str).eq("2021").all()
deprivation_support = deprivation_raw.loc[
    deprivation_raw["geography code"].isin(coverage_codes)
].copy()
assert set(deprivation_support["geography code"]) == coverage_codes, "Missing coverage areas"

print(f"Geographically selected source: {len(deprivation_support):,} areas, {len(deprivation_support.columns)} fields")
field_meanings = [
    "Census year — 2021 for every row",
    "Original statistical area name",
    "MSOA code — my link to HLE and the map",
    "Total households — denominator for percentages",
    "Households deprived in none of the four dimensions",
    "Households deprived in exactly one dimension",
    "Households deprived in exactly two dimensions",
    "Households deprived in exactly three dimensions",
    "Households deprived in all four dimensions",
]
assert len(field_meanings) == len(deprivation_support.columns)
display(pd.DataFrame({
    "Source field": deprivation_support.columns,
    "What it contains": field_meanings,
}))

# Preview Birmingham areas in source order; this is not a deprivation ranking.
# Transpose only the preview so every source field remains visible.
birmingham_deprivation = deprivation_support.loc[
    deprivation_support["geography"].str.startswith("Birmingham", na=False)
]
source_preview = birmingham_deprivation.head(5).set_index("geography code").T
source_preview.index.name = "Source field"
display(source_preview)


Geographically selected source: 660 areas, 9 fields


,Source field,What it contains
0,date,Census year — 2021 for every row
1,geography,Original statistical area name
2,geography code,MSOA code — my link to HLE and the map
3,Household deprivation: Total: All households; ...,Total households — denominator for percentages
4,Household deprivation: Household is not depriv...,Households deprived in none of the four dimens...
5,Household deprivation: Household is deprived i...,Households deprived in exactly one dimension
6,Household deprivation: Household is deprived i...,Households deprived in exactly two dimensions
7,Household deprivation: Household is deprived i...,Households deprived in exactly three dimensions
8,Household deprivation: Household is deprived i...,Households deprived in all four dimensions


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Household deprivation: Total: All households; measures: Value,2634,3130,3569,3988,4493
Household deprivation: Household is not deprived in any dimension; measures: Value,1433,1664,2244,2468,1994
Household deprivation: Household is deprived in one dimension; measures: Value,818,1031,1035,1139,1484
Household deprivation: Household is deprived in two dimensions; measures: Value,327,347,243,321,767
Household deprivation: Household is deprived in three dimensions; measures: Value,55,83,44,59,229
Household deprivation: Household is deprived in four dimensions; measures: Value,1,5,3,1,19


### What each field is

Date is 2021 throughout, so I state it here and drop the column. Geography is the numbered ONS name, which I replace. The code is my link to everything else. Total households is the denominator. The five deprivation columns count households deprived in zero, one, two, three or four dimensions. They say how many dimensions apply, not which ones.

I use the share of households deprived in two or more dimensions. It picks out areas where several kinds of disadvantage land on the same households. The category counts stay in the supporting file in case I change my mind.

### The measure

I add the households deprived in two, three or four dimensions, divide by total households and multiply by 100. A percentage lets me compare areas of different sizes fairly. The calculation doesn't say which dimensions apply, and the overlap with HLE's health question still matters.

In [9]:
# Use this source's household denominator, not a population total from another table.
total_field = "Household deprivation: Total: All households; measures: Value"
deprived_fields = [
    f"Household deprivation: Household is deprived in {dimension} dimensions; measures: Value"
    for dimension in ["two", "three", "four"]
]
counts = deprivation_support[[total_field, *deprived_fields]]
assert counts.notna().all().all(), "Missing household counts need review"
assert counts.ge(0).all().all(), "Negative household counts"
assert counts[total_field].gt(0).all(), "Invalid household denominator"
deprived_2plus = counts[deprived_fields].sum(axis=1)
assert deprived_2plus.le(counts[total_field]).all()

deprivation_display = pd.DataFrame({
    "area_code": deprivation_support["geography code"],
    "local_area_name": deprivation_support["geography code"].map(local_name_by_code),
    "households_deprived_2plus_pct": 100 * deprived_2plus / counts[total_field],
})
assert deprivation_display["local_area_name"].notna().all()
assert deprivation_display["area_code"].is_unique
assert deprivation_display["households_deprived_2plus_pct"].between(0, 100).all()
print(f"Prepared: {len(deprivation_display):,} areas; {len(deprivation_display.columns)} compact fields")
display(deprivation_display.loc[
    deprivation_display["area_code"].isin(birmingham_deprivation["geography code"])
].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Prepared: 660 areas; 3 compact fields


,area_code,local_area_name,households_deprived_2plus_pct
2922,E02001827,Hill Hook,14.54
2923,E02001828,Four Oaks,13.90
2924,E02001829,Little Sutton,8.13
2925,E02001830,Sutton Coldfield North,9.55
2926,E02001831,Reddicap Heath,22.59
2927,E02001832,Rectory Park,12.18
2928,E02001833,Kingstanding North,13.36
2929,E02001834,Old Oscott,25.66
2930,E02001835,Sutton Coldfield South,12.18
2931,E02001836,New Oscott,13.21


Preview rounded only; saved percentages retain precision.


### Save household deprivation

The compact table goes in `Data/Cleaned/Household Deprivation/`, with the source counts in a second file. Existing files are backed up on a rerun, and every saved file is read back and checked.

In [10]:
deprivation_folder = project_data("Cleaned/Household Deprivation")
deprivation_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("household_deprivation_2021_msoa.csv", deprivation_display),
    ("household_deprivation_2021_msoa_support.csv", deprivation_support),
]:
    destination = deprivation_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Household Deprivation"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows, {len(reloaded.columns)} fields")

Saved and verified household_deprivation_2021_msoa.csv: 660 rows, 3 fields
Saved and verified household_deprivation_2021_msoa_support.csv: 660 rows, 9 fields


Across my 660 areas, the share of households deprived in two or more dimensions runs from 6.1% (Monkspath North) to 41.5% (Sparkbrook South). The median is 19.3%.

## 3. Overcrowding

Census 2021 table TS052, occupancy rating for bedrooms. It compares the bedrooms a household has with the number it needs under the Census bedroom standard, so it isn't simply people per bedroom. Housing conditions may link to health, but I haven't shown that they do in these areas. The measure only covers a shortfall in bedrooms. It says nothing about damp, repairs or any other part of housing quality.

Same steps as before: read the MSOA file from its ZIP and pick my 660 areas, keeping every column for now.

In [11]:
overcrowding_zip = raw_file("census2021-ts052.zip")
with ZipFile(overcrowding_zip) as archive:
    with archive.open("census2021-ts052-msoa.csv") as source_file:
        overcrowding_raw = pd.read_csv(source_file)
assert overcrowding_raw["geography code"].is_unique
assert overcrowding_raw["date"].astype(str).eq("2021").all()

# Filter geography first; leave all source columns available for inspection.
overcrowding_support = overcrowding_raw.loc[
    overcrowding_raw["geography code"].isin(coverage_codes)
].copy()
assert set(overcrowding_support["geography code"]) == coverage_codes
print(f"National source: {len(overcrowding_raw):,} areas")
print(f"Selected coverage: {len(overcrowding_support):,} areas; {len(overcrowding_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 9 fields


### The fields

Positive ratings mean spare bedrooms, zero means the household has exactly what it needs, and negative ratings mean a shortfall. Zero is a real category, not missing data. The sample is five Birmingham areas in source order.

In [12]:
overcrowding_meanings = [
    "Census year — 2021 throughout",
    "Original statistical area name",
    "MSOA code — link to HLE and the map",
    "Total households — denominator for percentages",
    "Households with at least two bedrooms above the standard",
    "Households with one bedroom above the standard",
    "Households with exactly the number of bedrooms required",
    "Households with one fewer bedroom than required",
    "Households with at least two fewer bedrooms than required",
]
assert len(overcrowding_meanings) == len(overcrowding_support.columns)
display(pd.DataFrame({"Source field": overcrowding_support.columns,
                      "What it contains": overcrowding_meanings}))
# Transpose the preview so the full set of fields fits vertically.
birmingham_overcrowding = overcrowding_support.loc[
    overcrowding_support["geography"].str.startswith("Birmingham", na=False)
]
overcrowding_preview = birmingham_overcrowding.head(5).set_index("geography code").T
overcrowding_preview.index.name = "Source field"
display(overcrowding_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Occupancy rating for bedrooms: Total: All hous...,Total households — denominator for percentages
4,Occupancy rating for bedrooms: Occupancy ratin...,Households with at least two bedrooms above th...
5,Occupancy rating for bedrooms: Occupancy ratin...,Households with one bedroom above the standard
6,Occupancy rating for bedrooms: Occupancy ratin...,Households with exactly the number of bedrooms...
7,Occupancy rating for bedrooms: Occupancy ratin...,Households with one fewer bedroom than required
8,Occupancy rating for bedrooms: Occupancy ratin...,Households with at least two fewer bedrooms th...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Occupancy rating for bedrooms: Total: All households,2633,3130,3571,3987,4493
Occupancy rating for bedrooms: Occupancy rating of bedrooms: +2 or more,1229,1307,2000,2351,1482
Occupancy rating for bedrooms: Occupancy rating of bedrooms: +1,865,1166,1109,1027,1474
Occupancy rating for bedrooms: Occupancy rating of bedrooms: 0,490,605,417,546,1329
Occupancy rating for bedrooms: Occupancy rating of bedrooms: -1,43,46,39,55,184
Occupancy rating for bedrooms: Occupancy rating of bedrooms: -2 or less,6,6,6,8,24


### The measure

I keep households with a bedroom shortfall (%): households at -1 and at -2 or less, divided by total households, times 100. The compact table has the code, local name and that percentage. The spare-bedroom columns stay in the supporting file, and I can go back to them if bedroom occupancy turns out to matter for HLE.

In [13]:
bedroom_total = "Occupancy rating for bedrooms: Total: All households"
shortfall_fields = [
    "Occupancy rating for bedrooms: Occupancy rating of bedrooms: -1",
    "Occupancy rating for bedrooms: Occupancy rating of bedrooms: -2 or less",
]
# Use the matching household denominator, not a population count.
bedroom_counts = overcrowding_support[[bedroom_total, *shortfall_fields]]
assert bedroom_counts.notna().all().all(), "Missing counts need review"
assert bedroom_counts.ge(0).all().all(), "Negative household counts"
assert bedroom_counts[bedroom_total].gt(0).all(), "Invalid household denominator"
shortfall_count = bedroom_counts[shortfall_fields].sum(axis=1)
assert shortfall_count.le(bedroom_counts[bedroom_total]).all()

overcrowding_display = pd.DataFrame({
    "area_code": overcrowding_support["geography code"],
    "local_area_name": overcrowding_support["geography code"].map(local_name_by_code),
    "overcrowded_households_pct": 100 * shortfall_count / bedroom_counts[bedroom_total],
})
assert overcrowding_display["area_code"].is_unique
assert overcrowding_display["local_area_name"].notna().all()
assert overcrowding_display["overcrowded_households_pct"].between(0, 100).all()
print(f"Compact table: {len(overcrowding_display):,} areas; {len(overcrowding_display.columns)} fields")
display(overcrowding_display.loc[
    overcrowding_display["area_code"].isin(birmingham_overcrowding["geography code"])
].head(10).round(2))

Compact table: 660 areas; 3 fields


,area_code,local_area_name,overcrowded_households_pct
1746,E02001827,Hill Hook,1.86
1747,E02001828,Four Oaks,1.66
1748,E02001829,Little Sutton,1.26
1749,E02001830,Sutton Coldfield North,1.58
1750,E02001831,Reddicap Heath,4.63
1751,E02001832,Rectory Park,1.89
1752,E02001833,Kingstanding North,2.49
1753,E02001834,Old Oscott,5.73
1754,E02001835,Sutton Coldfield South,1.81
1755,E02001836,New Oscott,1.67


### Save overcrowding

The three-column table goes in `Data/Cleaned/Overcrowding/`, with the nine-field counts in a supporting file. Backed up on rerun, then read back and checked.

In [14]:
overcrowding_folder = project_data("Cleaned/Overcrowding")
overcrowding_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("overcrowding_2021_msoa.csv", overcrowding_display),
    ("overcrowding_2021_msoa_support.csv", overcrowding_support),
]:
    destination = overcrowding_folder / filename
    # Preserve the previous export before replacing it.
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Overcrowding"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified overcrowding_2021_msoa.csv: 660 rows; 3 fields
Saved and verified overcrowding_2021_msoa_support.csv: 660 rows; 9 fields


Overcrowding runs from 0.5% of households (Weston) to 29.0% (Saltley East). The median is only 2.9%, so most areas are low and a few are very high.

## 4. Fuel poverty

The 2024 fuel poverty estimates, published in 2026. I use Table 4, which is by LSOA. LSOAs are smaller than the MSOAs in my HLE data, so I have to combine them into their parent MSOAs.

A home that's hard to keep warm may link to health. These are the source's fuel poverty estimates, not unpaid bills. They're also from 2024, which is after the 2019–2023 HLE period, so any link I find has to be read with that in mind, but also things like fuel poverty do not change majorly over such a short time without some form of crisis intervening. So for my purposes, I regard the data as valid and of utility. The workbook warns that small-area figures are good for general patterns and not for tiny differences between similar areas. Combining them doesn't remove that.

The justification for that: the government's fuel poverty factsheet for England says "After remaining stable from 2019, the proportion of fuel poor households has decreased since 2022" (Fuel Poverty Factsheet, England, 2024 statistics, section "Fuel poverty in 2024": <https://assets.publishing.service.gov.uk/media/67e3d2ebd4a1b0665b8ee2c3/Factsheet_2025.pdf>). That is a national figure for England, so it doesn't prove the local pattern held, but it points the same way. For pollution and crime I'm making the same assumption without a source.


In [15]:
fuel_workbook = raw_file("fuel_poverty_2024_tables_published_2026.xlsx")
# Table 4 headings are on row 3; exclude titles and footer rows using valid area codes.
fuel_sheet = pd.read_excel(fuel_workbook, sheet_name="Table 4", header=2)
fuel_raw = fuel_sheet.loc[
    fuel_sheet["LSOA Code"].astype("string").str.fullmatch(r"E010\d{5}", na=False)
].copy()
assert fuel_raw["LSOA Code"].is_unique

lookup_file = project_data("Prepared/Geography/ons_lsoa21_to_msoa21_lad22_exploratory_west_midlands.csv")
fuel_lookup = pd.read_csv(lookup_file)
fuel_lookup = fuel_lookup.loc[fuel_lookup["MSOA21CD"].isin(coverage_codes)].copy()
assert fuel_lookup["LSOA21CD"].is_unique
assert set(fuel_lookup["MSOA21CD"]) == coverage_codes
# Require every expected smaller area: never silently calculate a partial MSOA.
missing_lsoas = set(fuel_lookup["LSOA21CD"]) - set(fuel_raw["LSOA Code"])
assert not missing_lsoas, f"Expected LSOAs missing from fuel poverty source: {sorted(missing_lsoas)}"
fuel_selected = fuel_raw.loc[fuel_raw["LSOA Code"].isin(fuel_lookup["LSOA21CD"])].copy()
print(f"National source: {len(fuel_raw):,} LSOAs")
print(f"Selected source: {len(fuel_selected):,} LSOAs; {len(fuel_selected.columns)} original fields")
print(f"Parent MSOAs: {fuel_lookup['MSOA21CD'].nunique():,}; missing expected LSOAs: {len(missing_lsoas)}")

National source: 33,755 LSOAs
Selected source: 3,205 LSOAs; 8 original fields
Parent MSOAs: 660; missing expected LSOAs: 0


### The fields

Eight fields: LSOA code and name, local authority code and name, region, number of households, number in fuel poverty, and the published percentage. My lookup adds the MSOA code I need to combine the LSOAs. That's supporting geography and not another feature. The sample is five Birmingham LSOAs in source order.

In [16]:
fuel_meanings = [
    "LSOA code — match to the MSOA lookup",
    "Original LSOA name",
    "Source local-authority code",
    "Source local-authority name",
    "Source region label",
    "Estimated total households — denominator",
    "Estimated households in fuel poverty — numerator",
    "Published LSOA fuel-poverty percentage",
]
assert len(fuel_meanings) == len(fuel_selected.columns)
display(pd.DataFrame({"Source field": fuel_selected.columns, "What it contains": fuel_meanings}))
fuel_birmingham_preview = fuel_selected.loc[fuel_selected["Local Authority Name"].eq("Birmingham")].head(5).set_index("LSOA Code").T
fuel_birmingham_preview.index.name = "Source field"
display(fuel_birmingham_preview)

,Source field,What it contains
0,LSOA Code,LSOA code — match to the MSOA lookup
1,LSOA Name,Original LSOA name
2,Local Authority Code,Source local-authority code
3,Local Authority Name,Source local-authority name
4,Region,Source region label
5,Number of households,Estimated total households — denominator
6,Number of households in fuel poverty,Estimated households in fuel poverty — numerator
7,Proportion of households fuel poor (%),Published LSOA fuel-poverty percentage


LSOA Code,E01008881,E01008882,E01008883,E01008884,E01008885
Source field,,,,,
LSOA Name,Birmingham 067A,Birmingham 066A,Birmingham 078A,Birmingham 078B,Birmingham 076A
Local Authority Code,E08000025,E08000025,E08000025,E08000025,E08000025
Local Authority Name,Birmingham,Birmingham,Birmingham,Birmingham,Birmingham
Region,West Midlands,West Midlands,West Midlands,West Midlands,West Midlands
Number of households,612.0,696.0,664.0,740.0,525.0
Number of households in fuel poverty,88.0,114.0,112.0,175.0,54.0
Proportion of households fuel poor (%),14.379085,16.37931,16.86747,23.648649,10.285714


### Combine LSOAs into MSOAs

For each MSOA I add up the fuel-poor households and the total households across its LSOAs, then take 100 × fuel-poor ÷ total. I don't average the LSOA percentages, because a small LSOA and a big one would count equally.

The workbook uses May 2026 geography codes and my bridge uses 2021 LSOA and MSOA codes. So the code checks that every LSOA has exactly one parent and that none is missing, and doesn't assume the two match. The counts are rounded estimates, which makes the result my own summary and not an official MSOA statistic.

In [17]:
fuel_lsoa_support = fuel_selected.merge(
    fuel_lookup[["LSOA21CD", "MSOA21CD"]],
    left_on="LSOA Code", right_on="LSOA21CD", how="left", validate="one_to_one",
)
assert len(fuel_lsoa_support) == len(fuel_selected)
assert fuel_lsoa_support["MSOA21CD"].notna().all()
fuel_count_fields = ["Number of households", "Number of households in fuel poverty"]
for field in [*fuel_count_fields, "Proportion of households fuel poor (%)"]:
    fuel_lsoa_support[field] = pd.to_numeric(fuel_lsoa_support[field], errors="raise")
assert fuel_lsoa_support[fuel_count_fields].notna().all().all()
assert fuel_lsoa_support[fuel_count_fields].ge(0).all().all()
assert fuel_lsoa_support["Number of households"].gt(0).all()
assert fuel_lsoa_support["Number of households in fuel poverty"].le(fuel_lsoa_support["Number of households"]).all()

# Sum numerators and denominators before calculating the MSOA percentage.
fuel_msoa_support = fuel_lsoa_support.groupby("MSOA21CD", as_index=False)[fuel_count_fields].sum()
fuel_msoa_support = fuel_msoa_support.rename(columns={"MSOA21CD": "area_code"})
fuel_msoa_support["lsoa_count"] = fuel_msoa_support["area_code"].map(fuel_lsoa_support.groupby("MSOA21CD").size())
fuel_display = pd.DataFrame({
    "area_code": fuel_msoa_support["area_code"],
    "local_area_name": fuel_msoa_support["area_code"].map(local_name_by_code),
    "fuel_poor_households_pct": 100 * fuel_msoa_support["Number of households in fuel poverty"] / fuel_msoa_support["Number of households"],
})
assert set(fuel_display["area_code"]) == coverage_codes
assert fuel_display["area_code"].is_unique
assert fuel_display["local_area_name"].notna().all()
assert fuel_display["fuel_poor_households_pct"].between(0, 100).all()
print(f"Compact table: {len(fuel_display):,} MSOAs; {len(fuel_display.columns)} fields")
birmingham_codes = set(area_code for area_code, entry in local_names.items() if entry["authority"] == "Birmingham")
display(fuel_display.loc[fuel_display["area_code"].isin(birmingham_codes)].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 MSOAs; 3 fields


,area_code,local_area_name,fuel_poor_households_pct
0,E02001827,Hill Hook,7.13
1,E02001828,Four Oaks,8.43
2,E02001829,Little Sutton,7.10
3,E02001830,Sutton Coldfield North,7.27
4,E02001831,Reddicap Heath,9.02
5,E02001832,Rectory Park,6.83
6,E02001833,Kingstanding North,10.06
7,E02001834,Old Oscott,13.16
8,E02001835,Sutton Coldfield South,10.28
9,E02001836,New Oscott,8.70


Preview rounded only; saved percentages retain precision.


### Save fuel poverty

The compact table goes in `Data/Cleaned/Fuel Poverty/`, with an LSOA file and an MSOA file of source counts, area relationships and the number of LSOAs combined. Backed up on rerun, then read back and checked.

In [18]:
fuel_folder = project_data("Cleaned/Fuel Poverty")
fuel_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("fuel_poverty_2024_msoa.csv", fuel_display),
    ("fuel_poverty_2024_msoa_support.csv", fuel_msoa_support),
    ("fuel_poverty_2024_lsoa_support.csv", fuel_lsoa_support),
]:
    destination = fuel_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Fuel Poverty"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified fuel_poverty_2024_msoa.csv: 660 rows; 3 fields
Saved and verified fuel_poverty_2024_msoa_support.csv: 660 rows; 4 fields
Saved and verified fuel_poverty_2024_lsoa_support.csv: 3,205 rows; 10 fields


Fuel poverty runs from 4.0% of households (Monkspath North) to 28.2% (Lower Stoke). The median is 11.1%.

## 5. Pollution

Two 2024 annual-mean background estimates from Defra: nitrogen dioxide (NO₂) and fine particles (PM2.5), both in micrograms per cubic metre (µg/m³). PM2.5 means particles up to 2.5 micrometres across.

These come as 1 km by 1 km grid squares, not as statistical areas. Each file has a grid code, the centre point of the square (`x` and `y`, on the British National Grid) and a value. I need a spatial calculation to turn them into MSOA figures.

They're modelled background levels, not readings at every address, and they don't capture roadside conditions. They're also from 2024, after the HLE period. [Defra's documentation](https://uk-air.defra.gov.uk/data/pcm-data) explains the grid and the `MISSING` marker, which I read as missing and not as zero. The first five rows of each file are intro text, so `skiprows=5` starts at the real header.

In [19]:
no2_raw = pd.read_csv(raw_file("defra_no2_2024_1km.csv"), skiprows=5, na_values=["MISSING"])
pm25_raw = pd.read_csv(raw_file("defra_pm25_2024_1km.csv"), skiprows=5, na_values=["MISSING"])
for label, table, value_field in [("NO2", no2_raw, "no22024"), ("PM2.5", pm25_raw, "pm252024g")]:
    assert table["gridcode"].is_unique
    assert not table.duplicated(["x", "y"]).any()
    table[value_field] = pd.to_numeric(table[value_field], errors="raise")
    print(f"{label}: {len(table):,} grid cells; {len(table.columns)} fields; {table[value_field].isna().sum()} unavailable values nationally")
    display(pd.DataFrame({"Source field": table.columns, "What it contains": [
        "Grid-square identifier — not an MSOA code",
        "Grid centre easting in metres",
        "Grid centre northing in metres",
        "2024 annual-mean concentration in micrograms per cubic metre",
    ]}))

NO2: 254,905 grid cells; 4 fields; 0 unavailable values nationally


,Source field,What it contains
0,gridcode,Grid-square identifier — not an MSOA code
1,x,Grid centre easting in metres
2,y,Grid centre northing in metres
3,no22024,2024 annual-mean concentration in micrograms p...


PM2.5: 254,905 grid cells; 4 fields; 1 unavailable values nationally


,Source field,What it contains
0,gridcode,Grid-square identifier — not an MSOA code
1,x,Grid centre easting in metres
2,y,Grid centre northing in metres
3,pm252024g,2024 annual-mean concentration in micrograms p...


### Pick the grid squares near my areas

I keep squares that could overlap the bounding box of my 660 MSOAs, plus half a kilometre each side, because the coordinates are square centres and not corners. A rectangle takes in some squares outside the real boundaries, so the proper spatial matching has to remove those. The preview uses grid centres inside Birmingham's boundaries so the examples relate to the city. They aren't the most polluted places.

In [20]:
pollution_areas = area_boundaries.to_crs(27700)
xmin, ymin, xmax, ymax = pollution_areas.total_bounds
pollution_candidates = {}
for label, table in [("NO2", no2_raw), ("PM2.5", pm25_raw)]:
    candidate = table.loc[
        table["x"].between(xmin - 500, xmax + 500)
        & table["y"].between(ymin - 500, ymax + 500)
    ].copy()
    pollution_candidates[label] = candidate
    # Preview city grid centres only; the final MSOA method is still to be chosen.
    points = gpd.GeoDataFrame(candidate, geometry=gpd.points_from_xy(candidate["x"], candidate["y"]), crs=27700)
    city_areas = pollution_areas.loc[pollution_areas["MSOA21CD"].isin(birmingham_codes), ["geometry"]]
    city_points = gpd.sjoin(points, city_areas, how="inner", predicate="within")
    print(f"{label}: {len(candidate):,} candidate grid cells across the coverage extent")
    display(city_points.drop(columns=["geometry", "index_right"]).head(5))

NO2: 14,630 candidate grid cells across the coverage extent


,gridcode,x,y,no22024
168345,689731,410500,300500,10.233360
168346,689732,411500,300500,9.365366
168347,689733,412500,300500,9.100857
168741,690420,409500,299500,10.759030
168742,690421,410500,299500,10.736290


PM2.5: 14,630 candidate grid cells across the coverage extent


,gridcode,x,y,pm252024g
168345,689731,410500,300500,7.289942
168346,689732,411500,300500,7.159149
168347,689733,412500,300500,7.092629
168741,690420,409500,299500,7.392168
168742,690421,410500,299500,7.415641


### Area-weighted mean

For each MSOA I take the grid squares that overlap it and weight each square's value by the area of the overlap inside the boundary: the sum of (concentration × overlap area) divided by the sum of valid overlap area. Each square counts as one constant value. Small MSOAs can end up sharing a value, because I can't get more detail than the grid.

This is the average modelled level across the area. It isn't what residents breathe, since fields and housing count by area and not by how many people live there. I check how much of each MSOA is covered by valid values, and I flag any area that isn't fully covered so a partial mean never passes as complete.

In [21]:
import numpy as np
from shapely.geometry import box

assert pollution_areas.geometry.is_valid.all(), "Invalid MSOA geometry needs review"
assert pollution_areas["MSOA21CD"].is_unique
pollution_tables = {}
pollution_checks = {}
pollution_overlaps = {}

for label, value_field, output_field in [
    ("NO2", "no22024", "no2_area_mean_ug_m3"),
    ("PM2.5", "pm252024g", "pm25_area_mean_ug_m3"),
]:
    candidate = pollution_candidates[label].copy()
    # The supplied coordinates are square centres: construct 1 km cells around them.
    cells = gpd.GeoDataFrame(candidate, geometry=[
        box(x - 500, y - 500, x + 500, y + 500)
        for x, y in zip(candidate["x"], candidate["y"])
    ], crs=27700)
    # Clip cells to MSOA polygons; only the intersection area supplies weight.
    overlap = gpd.overlay(
        pollution_areas[["MSOA21CD", "geometry"]], cells,
        how="intersection", keep_geom_type=True,
    )
    overlap["overlap_m2"] = overlap.geometry.area
    overlap = overlap.loc[overlap["overlap_m2"].gt(0)].copy()
    valid = overlap.loc[overlap[value_field].notna()].copy()
    assert valid[value_field].ge(0).all(), "Negative pollution concentration"
    valid["weighted_concentration"] = valid[value_field] * valid["overlap_m2"]
    sums = valid.groupby("MSOA21CD")[["weighted_concentration", "overlap_m2"]].sum()
    means = sums["weighted_concentration"] / sums["overlap_m2"]

    checks = pollution_areas[["MSOA21CD"]].copy().rename(columns={"MSOA21CD": "area_code"})
    checks["msoa_area_m2"] = pollution_areas.geometry.area.to_numpy()
    checks["grid_overlap_m2"] = checks["area_code"].map(overlap.groupby("MSOA21CD")["overlap_m2"].sum()).fillna(0)
    checks["valid_overlap_m2"] = checks["area_code"].map(sums["overlap_m2"]).fillna(0)
    checks["valid_coverage_pct"] = 100 * checks["valid_overlap_m2"] / checks["msoa_area_m2"]
    checks["intersecting_cells"] = checks["area_code"].map(overlap.groupby("MSOA21CD")["gridcode"].nunique()).fillna(0).astype(int)
    # Allow only a tiny numerical tolerance for geometric calculations.
    assert checks["valid_coverage_pct"].le(100.0001).all(), "Overlapping grid cells counted twice"
    checks["complete_valid_coverage"] = np.isclose(checks["valid_overlap_m2"], checks["msoa_area_m2"], rtol=1e-6, atol=1)
    result = pd.DataFrame({
        "area_code": checks["area_code"],
        "local_area_name": checks["area_code"].map(local_name_by_code),
        output_field: checks["area_code"].map(means),
    })
    # Preserve missingness for any area without complete valid coverage.
    result.loc[~checks["complete_valid_coverage"], output_field] = np.nan
    assert result["area_code"].is_unique and set(result["area_code"]) == coverage_codes
    assert result["local_area_name"].notna().all()
    pollution_tables[label] = result
    pollution_checks[label] = checks
    pollution_overlaps[label] = overlap.drop(columns="geometry")
    print(f"{label}: {len(result):,} MSOAs; {len(result.columns)} compact fields")
    print(f"Complete valid coverage: {checks['complete_valid_coverage'].sum():,}; unavailable area means: {result[output_field].isna().sum()}")
    display(result.loc[result["area_code"].isin(birmingham_codes)].head(10).round(2))

NO2: 660 MSOAs; 3 compact fields
Complete valid coverage: 660; unavailable area means: 0


,area_code,local_area_name,no2_area_mean_ug_m3
0,E02001827,Hill Hook,10.11
1,E02001828,Four Oaks,10.44
2,E02001829,Little Sutton,10.43
3,E02001830,Sutton Coldfield North,11.43
4,E02001831,Reddicap Heath,11.37
5,E02001832,Rectory Park,12.50
6,E02001833,Kingstanding North,13.17
7,E02001834,Old Oscott,14.22
8,E02001835,Sutton Coldfield South,12.81
9,E02001836,New Oscott,13.80


PM2.5: 660 MSOAs; 3 compact fields
Complete valid coverage: 660; unavailable area means: 0


,area_code,local_area_name,pm25_area_mean_ug_m3
0,E02001827,Hill Hook,7.29
1,E02001828,Four Oaks,7.32
2,E02001829,Little Sutton,7.30
3,E02001830,Sutton Coldfield North,7.48
4,E02001831,Reddicap Heath,7.46
5,E02001832,Rectory Park,7.58
6,E02001833,Kingstanding North,7.88
7,E02001834,Old Oscott,8.09
8,E02001835,Sutton Coldfield South,7.68
9,E02001836,New Oscott,7.92


### Save pollution

Two three-column tables, one for NO₂ and one for PM2.5, with the coverage checks and grid overlaps in supporting files. The previews are rounded for reading, and the saved values keep full precision.

In [22]:
pollution_folder = project_data("Cleaned/Pollution")
pollution_folder.mkdir(parents=True, exist_ok=True)
for label, stem in [("NO2", "no2_2024"), ("PM2.5", "pm25_2024")]:
    for filename, table in [
        (f"{stem}_msoa.csv", pollution_tables[label]),
        (f"{stem}_msoa_coverage_checks.csv", pollution_checks[label]),
        (f"{stem}_grid_overlap_support.csv", pollution_overlaps[label]),
    ]:
        destination = pollution_folder / filename
        if destination.exists():
            backup_folder = project_folder / "VS_CODE/Backups/Outputs/Pollution"
            backup_folder.mkdir(parents=True, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
            shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
        table.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
        print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified no2_2024_msoa.csv: 660 rows; 3 fields
Saved and verified no2_2024_msoa_coverage_checks.csv: 660 rows; 7 fields
Saved and verified no2_2024_grid_overlap_support.csv: 16,312 rows; 6 fields
Saved and verified pm25_2024_msoa.csv: 660 rows; 3 fields
Saved and verified pm25_2024_msoa_coverage_checks.csv: 660 rows; 7 fields
Saved and verified pm25_2024_grid_overlap_support.csv: 16,312 rows; 6 fields


All 660 areas are fully covered. NO₂ runs from 2.5 µg/m³ (Wigmore) to 21.6 (Birmingham city centre), with a median of 10.3. PM2.5 varies much less, from 4.8 (Wigmore) to 8.7 (Sparkhill North), with a median of 7.3.

## 6. Economic activity

Census 2021 table TS066, economic activity status, for residents aged 16 and over. It covers people in work, unemployed people and economically inactive people, so it tells me about taking part in the labour market. It doesn't say whether jobs are secure, well paid or hazardous. The 16+ group includes retired people, so an employment share here isn't a working-age employment rate.

I read the MSOA file from its ZIP, pick my 660 areas and keep all the columns.

In [23]:
activity_zip = raw_file("census2021-ts066.zip")
with ZipFile(activity_zip) as archive:
    with archive.open("census2021-ts066-msoa.csv") as source_file:
        activity_raw = pd.read_csv(source_file)
assert activity_raw["geography code"].is_unique
assert activity_raw["date"].astype(str).eq("2021").all()
activity_support = activity_raw.loc[activity_raw["geography code"].isin(coverage_codes)].copy()
assert set(activity_support["geography code"]) == coverage_codes
print(f"National source: {len(activity_raw):,} areas")
print(f"Selected coverage: {len(activity_support):,} areas; {len(activity_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 34 fields


### The fields

There are 34 fields, with a lot of nesting. Employment splits into employees and self-employed, and those split again into full-time and part-time. I mustn't add a parent total to its children, or I'd count people twice. Full-time students are listed separately from other economically active residents. Students can be employed or unemployed too, so I need both branches for any measure that covers everyone. The sample is five Birmingham areas in source order.

In [24]:
activity_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original statistical area name",
    "geography code": "MSOA code — link to HLE and the map",
}
display(pd.DataFrame({
    "Source field": activity_support.columns,
    "What it contains": [activity_identifiers.get(field, "Count of residents aged 16+: " + field.removeprefix("Economic activity status: ")) for field in activity_support.columns],
}))
birmingham_activity = activity_support.loc[activity_support["geography"].str.startswith("Birmingham", na=False)]
# Turn the preview sideways so all the long field names remain visible.
activity_preview = birmingham_activity.head(5).set_index("geography code").T
activity_preview.index.name = "Source field"
display(activity_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Economic activity status: Total: All usual res...,Count of residents aged 16+: Total: All usual ...
4,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
5,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
6,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
7,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
8,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
9,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Economic activity status: Total: All usual residents aged 16 years and over,5188,5859,7139,8122,8333
Economic activity status: Economically active (excluding full-time students),2877,3173,4403,4516,4946
Economic activity status: Economically active (excluding full-time students):In employment,2763,3048,4259,4396,4650
Economic activity status: Economically active (excluding full-time students):In employment:Employee,2278,2484,3561,3497,4043
Economic activity status: Economically active (excluding full-time students): In employment: Employee: Part-time,555,666,888,884,1020
Economic activity status: Economically active (excluding full-time students): In employment: Employee: Full-time,1723,1818,2673,2613,3023
Economic activity status: Economically active (excluding full-time students):In employment:Self-employed with employees,99,139,152,259,73


### Two measures

This table is wide and deep, and taking every column would bury the question. I want a few clearly defined measures I can compare, and later use to group areas with machine learning.

- **Employed residents aged 16+ (%):** employed residents, including employed full-time students, divided by all residents aged 16+.
- **Unemployment rate (%):** unemployed residents, including unemployed full-time students, divided by all economically active residents.

They use different denominators, so one isn't the complement of the other. Economically inactive people (retired, studying, caring, ill) sit outside the unemployment rate. The detailed breakdowns stay in the supporting file. If I look at long-term sickness or disability later, I'll allow for how much it overlaps with what HLE measures.

In [25]:
activity_total_field = "Economic activity status: Total: All usual residents aged 16 years and over"
active_branches = [
    "Economic activity status: Economically active (excluding full-time students)",
    "Economic activity status: Economically active and a full-time student",
]
employment_fields = [branch + ":In employment" for branch in active_branches]
unemployment_fields = [branch + ": Unemployed" for branch in active_branches]
needed_activity_fields = [activity_total_field, *active_branches, *employment_fields, *unemployment_fields]
activity_counts = activity_support[needed_activity_fields]
assert activity_counts.notna().all().all(), "Missing counts need review"
assert activity_counts.ge(0).all().all(), "Negative resident counts"

# Include students consistently in both the numerators and relevant denominators.
residents_16plus = activity_counts[activity_total_field]
active_residents = activity_counts[active_branches].sum(axis=1)
employed_residents = activity_counts[employment_fields].sum(axis=1)
unemployed_residents = activity_counts[unemployment_fields].sum(axis=1)
assert residents_16plus.gt(0).all() and active_residents.gt(0).all()
assert active_residents.le(residents_16plus).all()
assert employed_residents.le(residents_16plus).all()
assert unemployed_residents.le(active_residents).all()

activity_display = pd.DataFrame({
    "area_code": activity_support["geography code"],
    "local_area_name": activity_support["geography code"].map(local_name_by_code),
    "employed_residents_16plus_pct": 100 * employed_residents / residents_16plus,
    "unemployment_rate_active_pct": 100 * unemployed_residents / active_residents,
})
assert activity_display["area_code"].is_unique
assert activity_display["local_area_name"].notna().all()
assert activity_display[["employed_residents_16plus_pct", "unemployment_rate_active_pct"]].ge(0).all().all()
assert activity_display[["employed_residents_16plus_pct", "unemployment_rate_active_pct"]].le(100).all().all()
print(f"Compact table: {len(activity_display):,} areas; {len(activity_display.columns)} fields")
display(activity_display.loc[activity_display["area_code"].isin(birmingham_activity["geography code"])].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 areas; 4 fields


,area_code,local_area_name,employed_residents_16plus_pct,unemployment_rate_active_pct
1746,E02001827,Hill Hook,54.59,4.90
1747,E02001828,Four Oaks,52.94,4.82
1748,E02001829,Little Sutton,60.71,3.67
1749,E02001830,Sutton Coldfield North,55.13,3.24
1750,E02001831,Reddicap Heath,57.16,6.53
1751,E02001832,Rectory Park,58.02,4.41
1752,E02001833,Kingstanding North,58.99,5.15
1753,E02001834,Old Oscott,56.78,8.08
1754,E02001835,Sutton Coldfield South,60.80,4.23
1755,E02001836,New Oscott,56.62,4.39


Preview rounded only; saved percentages retain precision.


### Save economic activity

The four-column table goes in `Data/Cleaned/Economic Activity/`, with the 34-field counts in a supporting file. Backed up on rerun, then read back and checked.

In [26]:
activity_folder = project_data("Cleaned/Economic Activity")
activity_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("economic_activity_2021_msoa.csv", activity_display),
    ("economic_activity_2021_msoa_support.csv", activity_support),
]:
    destination = activity_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Economic Activity"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified economic_activity_2021_msoa.csv: 660 rows; 4 fields
Saved and verified economic_activity_2021_msoa_support.csv: 660 rows; 34 fields


Employment shares run from 21.5% (Selly Oak) to 71.5% (Hawks Green), with a median of 55.8%. The two lowest areas are dominated by students. In Selly Oak, 9,756 of 15,230 residents aged 16+ are economically inactive students. North Central is similar (5,866 of 13,702), and it also has the highest unemployment rate (29.7%). Altogether 16 areas have an employment share under 40%.

So in some areas the employment share mostly tells me how many students live there. Section 11 deals with that.

## 7. Qualifications

Census 2021 table TS067, highest level of qualification, for residents aged 16 and over. It isn't about school performance. Qualifications are one part of an area's social profile. They don't tell me anyone's health and they won't explain an HLE difference by themselves. I want to see whether they line up with the HLE differences on my map, and with any groups of areas I find later with machine learning.

Same steps: read the MSOA file from its ZIP and keep my 660 areas.

In [27]:
qualifications_zip = raw_file("census2021-ts067.zip")
with ZipFile(qualifications_zip) as archive:
    with archive.open("census2021-ts067-msoa.csv") as source_file:
        qualifications_raw = pd.read_csv(source_file)
assert qualifications_raw["geography code"].is_unique
assert qualifications_raw["date"].astype(str).eq("2021").all()
qualifications_support = qualifications_raw.loc[
    qualifications_raw["geography code"].isin(coverage_codes)
].copy()
assert set(qualifications_support["geography code"]) == coverage_codes
print(f"National source: {len(qualifications_raw):,} areas")
print(f"Selected coverage: {len(qualifications_support):,} areas; {len(qualifications_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 11 fields


### The fields

These are counts of residents in each category, so bigger areas have bigger counts whatever their profile. Percentages of the table's 16+ total let me compare areas of different sizes. The sample is five Birmingham areas in source order.

In [28]:
qualification_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original statistical area name",
    "geography code": "MSOA code — link to HLE and the map",
}
display(pd.DataFrame({
    "Source field": qualifications_support.columns,
    "What it contains": [qualification_identifiers.get(field, "Resident count: " + field) for field in qualifications_support.columns],
}))
birmingham_qualifications = qualifications_support.loc[
    qualifications_support["geography"].str.startswith("Birmingham", na=False)
]
# A vertical preview keeps the long category labels readable.
qualifications_preview = birmingham_qualifications.head(5).set_index("geography code").T
qualifications_preview.index.name = "Source field"
display(qualifications_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Highest level of qualification: Total: All usu...,Resident count: Highest level of qualification...
4,Highest level of qualification: No qualifications,Resident count: Highest level of qualification...
5,Highest level of qualification: Level 1 and en...,Resident count: Highest level of qualification...
6,Highest level of qualification: Level 2 qualif...,Resident count: Highest level of qualification...
7,Highest level of qualification: Apprenticeship,Resident count: Highest level of qualification...
8,Highest level of qualification: Level 3 qualif...,Resident count: Highest level of qualification...
9,Highest level of qualification: Level 4 qualif...,Resident count: Highest level of qualification...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Highest level of qualification: Total: All usual residents aged 16 years and over,5184,5859,7134,8123,8332
Highest level of qualification: No qualifications,728,923,750,906,1641
Highest level of qualification: Level 1 and entry level qualifications,428,469,526,585,878
Highest level of qualification: Level 2 qualifications,718,784,945,915,1183
Highest level of qualification: Apprenticeship,215,248,314,304,431
Highest level of qualification: Level 3 qualifications,810,860,1146,1228,1434
Highest level of qualification: Level 4 qualifications and above,2165,2417,3302,4016,2558


### Three measures

I started with no qualifications and Level 4 and above, but those are only the two ends. I added Level 2 to cover the middle.

- **No qualifications (%):** the lower end.
- **Level 2 (%):** the middle. Broadly five or more GCSEs at grades 4–9, or equivalent.
- **Level 4 and above (%):** the higher end, from HNC level up to postgraduate degrees.

All three are divided by the same 16+ total. I use Level 2 itself and not "Level 2 or above", so the middle isn't mixed up with Level 4+. The three don't add up to 100%, because Level 1, apprenticeships, Level 3 and other qualifications aren't in them. Those counts stay in the supporting file. The compact table has five columns: code, local name and the three percentages.

Source definitions: [ONS highest level of qualification](https://www.ons.gov.uk/census/census2021dictionary/variablesbytopic/educationvariablescensus2021/highestlevelofqualification).

In [29]:
qualification_total = next(field for field in qualifications_support if ": Total:" in field)
no_qualification_field = next(field for field in qualifications_support if field.endswith(": No qualifications"))
level4_field = next(field for field in qualifications_support if field.endswith(": Level 4 qualifications and above"))
level2_field = next(field for field in qualifications_support if field.endswith(": Level 2 qualifications"))
qualification_counts = qualifications_support[[qualification_total, no_qualification_field, level2_field, level4_field]]
assert qualification_counts.notna().all().all(), "Missing counts need review"
assert qualification_counts.ge(0).all().all(), "Negative resident counts"
assert qualification_counts[qualification_total].gt(0).all(), "Invalid resident denominator"
# Use this table's own population total for both percentages.
for field in [no_qualification_field, level2_field, level4_field]:
    assert qualification_counts[field].le(qualification_counts[qualification_total]).all()
assert (qualification_counts[no_qualification_field] + qualification_counts[level2_field] + qualification_counts[level4_field]).le(qualification_counts[qualification_total]).all()
qualifications_display = pd.DataFrame({
    "area_code": qualifications_support["geography code"],
    "local_area_name": qualifications_support["geography code"].map(local_name_by_code),
    "no_qualifications_pct": 100 * qualification_counts[no_qualification_field] / qualification_counts[qualification_total],
    "level2_qualifications_pct": 100 * qualification_counts[level2_field] / qualification_counts[qualification_total],
    "level4plus_pct": 100 * qualification_counts[level4_field] / qualification_counts[qualification_total],
})
assert qualifications_display["area_code"].is_unique
assert qualifications_display["local_area_name"].notna().all()
for field in ["no_qualifications_pct", "level2_qualifications_pct", "level4plus_pct"]:
    assert qualifications_display[field].between(0, 100).all()
print(f"Compact table: {len(qualifications_display):,} areas; {len(qualifications_display.columns)} fields")
display(qualifications_display.loc[
    qualifications_display["area_code"].isin(birmingham_qualifications["geography code"])
].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 areas; 5 fields


,area_code,local_area_name,no_qualifications_pct,level2_qualifications_pct,level4plus_pct
1746,E02001827,Hill Hook,14.04,13.85,41.76
1747,E02001828,Four Oaks,15.75,13.38,41.25
1748,E02001829,Little Sutton,10.51,13.25,46.29
1749,E02001830,Sutton Coldfield North,11.15,11.26,49.44
1750,E02001831,Reddicap Heath,19.70,14.20,30.70
1751,E02001832,Rectory Park,14.05,12.92,41.94
1752,E02001833,Kingstanding North,17.55,13.58,32.21
1753,E02001834,Old Oscott,25.39,15.36,20.79
1754,E02001835,Sutton Coldfield South,13.30,12.76,46.91
1755,E02001836,New Oscott,15.48,11.85,41.70


Preview rounded only; saved percentages retain precision.


### Save qualifications

The five-column table goes in `Data/Cleaned/Qualifications/`, with the 11-field counts in a supporting file. Backed up on rerun, then read back and checked.

In [30]:
qualifications_folder = project_data("Cleaned/Qualifications")
qualifications_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("qualifications_2021_msoa.csv", qualifications_display),
    ("qualifications_2021_msoa_support.csv", qualifications_support),
]:
    destination = qualifications_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Qualifications"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified qualifications_2021_msoa.csv: 660 rows; 5 fields
Saved and verified qualifications_2021_msoa_support.csv: 660 rows; 11 fields


No qualifications runs from 5.4% (Selly Oak) to 38.3% (Friar Park). Level 4 and above runs from 12.7% (Friar Park) to 67.2% (Birmingham city centre).

Selly Oak is also the lowest on Level 2 (3.1%). That's the student effect again. In Selly Oak 67.6% of residents aged 16+ have Level 3 as their highest qualification, the highest share of any of my areas (the median is 16.6%). Level 3 isn't one of my three measures, so these students barely show up in any of them. Section 11 deals with this too.

## 8. Household income

Household income gives me financial context for the HLE differences. I want one useful measure of household resources and not four versions of the same story.

The ONS workbook covers April 2022 to March 2023 and has four modelled MSOA averages: total income, net income, and household-size-adjusted net income before and after housing costs. These are annual household figures. They aren't individual salaries and they aren't debt.

Same steps, with the same 660 areas.

In [31]:
income_workbook = raw_file("ons_small_area_income_fye2023.xlsx")
income_sheets = [
    "Total annual income", "Net annual income",
    "Net income before housing costs", "Net income after housing costs",
]
display(pd.DataFrame({
    "Source sheet": income_sheets,
    "Measure": [
        "Gross annual household income, including benefits",
        "Disposable annual household income after specified deductions",
        "Disposable income adjusted for household size/composition, before housing costs",
        "Disposable income adjusted for household size/composition, after housing costs",
    ],
}))
# The workbook has three introductory rows above the column headings.
income_raw = pd.read_excel(income_workbook, sheet_name="Net income after housing costs", header=3)
assert income_raw["MSOA code"].is_unique
income_support = income_raw.loc[income_raw["MSOA code"].isin(coverage_codes)].copy()
assert set(income_support["MSOA code"]) == coverage_codes
print(f"National source: {len(income_raw):,} MSOAs")
print(f"Selected coverage: {len(income_support):,} MSOAs; {len(income_support.columns)} fields")

,Source sheet,Measure
0,Total annual income,"Gross annual household income, including benefits"
1,Net annual income,Disposable annual household income after speci...
2,Net income before housing costs,Disposable income adjusted for household size/...
3,Net income after housing costs,Disposable income adjusted for household size/...


National source: 7,264 MSOAs
Selected coverage: 660 MSOAs; 10 fields


### The fields

Ten fields: area code and name, local authority and region codes and names, the income estimate, its upper and lower 95% confidence limits, and the width of the interval. The limits show how uncertain a modelled average is. They stay in the supporting table and out of the compact one. The sample is five Birmingham areas in source order.

In [32]:
income_field_notes = [
    "MSOA join key", "Original ONS area name",
    "Local authority identifier", "Local authority name",
    "Region identifier", "Region name",
    "Estimated annual equivalised disposable household income after housing costs (£)",
    "Upper 95% confidence limit (£)", "Lower 95% confidence limit (£)",
    "Width of the confidence interval (£)",
]
assert len(income_field_notes) == len(income_support.columns)
display(pd.DataFrame({"Source field": income_support.columns, "What it contains": income_field_notes}))
birmingham_income = income_support.loc[income_support["Local authority name"].eq("Birmingham")]
display(birmingham_income.head(5).set_index("MSOA code").T)

,Source field,What it contains
0,MSOA code,MSOA join key
1,MSOA name,Original ONS area name
2,Local authority code,Local authority identifier
3,Local authority name,Local authority name
4,Region code,Region identifier
5,Region name,Region name
6,Disposable (net) annual income after housing c...,Estimated annual equivalised disposable househ...
7,Upper confidence limit (£),Upper 95% confidence limit (£)
8,Lower confidence limit (£),Lower 95% confidence limit (£)
9,Confidence interval (£),Width of the confidence interval (£)


MSOA code,E02001827,E02001828,E02001829,E02001830,E02001831
MSOA name,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Local authority code,E08000025,E08000025,E08000025,E08000025,E08000025
Local authority name,Birmingham,Birmingham,Birmingham,Birmingham,Birmingham
Region code,E12000005,E12000005,E12000005,E12000005,E12000005
Region name,West Midlands,West Midlands,West Midlands,West Midlands,West Midlands
Disposable (net) annual income after housing costs (£),36785,36596,37480,43436,32675
Upper confidence limit (£),42015,41801,42953,49849,37451
Lower confidence limit (£),32205,32038,32705,37847,28507
Confidence interval (£),9810,9763,10248,12002,8944


### The measure

I use net annual household income after housing costs, adjusted for household size and make-up. ONS calls the adjustment *equivalisation*. £30,000 supporting one person goes further than £30,000 supporting a family, and the adjustment puts households on the same footing.

Housing costs here means rent and mortgage interest. It doesn't cover every household bill, and mortgage capital repayments aren't deducted. The compact table has the code, local name and annual equivalised income after housing costs, in pounds.

These are modelled area averages. They aren't what each household earns, and they don't show that income explains HLE. Source definitions: [ONS income estimates, financial year ending 2023](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/bulletins/smallareamodelbasedincomeestimates/financialyearending2023).

In [33]:
income_value_field = "Disposable (net) annual income after housing costs (£)"
income_numeric_fields = [income_value_field, "Lower confidence limit (£)", "Upper confidence limit (£)", "Confidence interval (£)"]
assert income_support[income_numeric_fields].notna().all().all()
assert income_support[income_numeric_fields].ge(0).all().all()
assert income_support["Lower confidence limit (£)"].le(income_support[income_value_field]).all()
assert income_support[income_value_field].le(income_support["Upper confidence limit (£)"]).all()
assert (income_support["Upper confidence limit (£)"] - income_support["Lower confidence limit (£)"]).eq(income_support["Confidence interval (£)"]).all()
income_display = pd.DataFrame({
    "area_code": income_support["MSOA code"],
    "local_area_name": income_support["MSOA code"].map(local_name_by_code),
    "annual_equivalised_income_after_housing_costs_gbp": income_support[income_value_field],
})
assert income_display["area_code"].is_unique
assert income_display["local_area_name"].notna().all()
print(f"Compact table: {len(income_display):,} areas; {len(income_display.columns)} fields")
display(income_display.loc[income_display["area_code"].isin(birmingham_income["MSOA code"])].head(10))

Compact table: 660 areas; 3 fields


,area_code,local_area_name,annual_equivalised_income_after_housing_costs_gbp
2922,E02001827,Hill Hook,36785
2923,E02001828,Four Oaks,36596
2924,E02001829,Little Sutton,37480
2925,E02001830,Sutton Coldfield North,43436
2926,E02001831,Reddicap Heath,32675
2927,E02001832,Rectory Park,37801
2928,E02001833,Kingstanding North,34826
2929,E02001834,Old Oscott,26665
2930,E02001835,Sutton Coldfield South,35867
2931,E02001836,New Oscott,37634


### Save income

The three-column table goes in `Data/Cleaned/Income/`, with the ten-field table in a supporting file. The raw workbook still has all four income measures if I want to change my choice. Backed up on rerun, then read back and checked.

In [34]:
income_folder = project_data("Cleaned/Income")
income_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("income_after_housing_costs_fye2023_msoa.csv", income_display),
    ("income_after_housing_costs_fye2023_msoa_support.csv", income_support),
]:
    destination = income_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Income"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified income_after_housing_costs_fye2023_msoa.csv: 660 rows; 3 fields
Saved and verified income_after_housing_costs_fye2023_msoa_support.csv: 660 rows; 10 fields


Income after housing costs runs from £18,375 (Sparkbrook North) to £44,332 (Monkspath North). The median is £31,132.

## 9. Ethnicity

I want cultural context alongside the socioeconomic data. A broad group like "Asian" hides real differences between Chinese, Bangladeshi, Pakistani and Indian residents, so I keep the detailed categories.

This is Census 2021 table TS021, self-reported ethnic group on 21 March 2021, which falls inside the HLE period. It doesn't measure religion, beliefs or behaviour, so I can't read Muslim, Hindu or Sikh populations from it. Religion would need its own dataset.

I first applied for the safeguarded modelled ethnicity data from GeoDS at UCL. The application needed an academic supervisor, and my course co-ordinator declined to act as one, so I didn't complete it. I use the open Census snapshot instead.

In [35]:
ethnicity_zip = raw_file("census2021-ts021.zip")
with ZipFile(ethnicity_zip) as archive:
    with archive.open("census2021-ts021-msoa.csv") as source_file:
        ethnicity_raw = pd.read_csv(source_file)
assert ethnicity_raw["geography code"].is_unique
assert ethnicity_raw["date"].astype(str).eq("2021").all()
ethnicity_support = ethnicity_raw.loc[ethnicity_raw["geography code"].isin(coverage_codes)].copy()
assert set(ethnicity_support["geography code"]) == coverage_codes
print(f"National source: {len(ethnicity_raw):,} MSOAs")
print(f"Selected coverage: {len(ethnicity_support):,} MSOAs; {len(ethnicity_support.columns)} fields")

National source: 7,264 MSOAs
Selected coverage: 660 MSOAs; 28 fields


### The categories

There are 28 fields: three identifiers, a total, five broad groups and 19 detailed categories. The broad groups are subtotals of the detailed ones, so adding both would count people twice. Larger counts can just mean a larger population, so I use percentages. The sample is five Birmingham areas in source order, turned sideways so the labels stay readable.

In [36]:
ethnicity_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original ONS area name",
    "geography code": "MSOA join key",
}
display(pd.DataFrame({
    "Source field": ethnicity_support.columns,
    "What it contains": [
        ethnicity_identifiers.get(field,
            "Total resident count" if field == "Ethnic group: Total: All usual residents"
            else "Broad-group subtotal" if field.count(":") == 1 else "Detailed-category resident count")
        for field in ethnicity_support.columns
    ],
}))
birmingham_ethnicity = ethnicity_support.loc[ethnicity_support["geography"].str.startswith("Birmingham", na=False)]
display(birmingham_ethnicity.head(5).set_index("geography code").T)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original ONS area name
2,geography code,MSOA join key
3,Ethnic group: Total: All usual residents,Total resident count
4,"Ethnic group: Asian, Asian British or Asian Welsh",Broad-group subtotal
5,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
6,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
7,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
8,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
9,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Ethnic group: Total: All usual residents,6450,7141,9160,9686,10682
"Ethnic group: Asian, Asian British or Asian Welsh",850,653,1160,1374,819
"Ethnic group: Asian, Asian British or Asian Welsh: Bangladeshi",24,20,38,48,43
"Ethnic group: Asian, Asian British or Asian Welsh: Chinese",78,80,142,102,87
"Ethnic group: Asian, Asian British or Asian Welsh: Indian",517,359,723,927,376
"Ethnic group: Asian, Asian British or Asian Welsh: Pakistani",126,126,177,220,190
"Ethnic group: Asian, Asian British or Asian Welsh: Other Asian",105,68,80,77,123
"Ethnic group: Black, Black British, Black Welsh, Caribbean or African",134,134,164,184,495


### Keep the detail as percentages

I keep the 19 detailed categories, each divided by the total usual residents. That gives 21 columns: code, local name and 19 percentages. It's much wider than the other tables because the detail is the point. The five broad subtotals stay out of the feature table because the detailed categories already contain them. All the original counts are in the supporting file.

These shares describe who lives in an area. They don't describe a single shared culture. Census disclosure protection can make published totals differ slightly from the sum of the categories. I use the published total and don't force the percentages to add to exactly 100%.

Source definitions: [ONS ethnic group, Census 2021](https://www.ons.gov.uk/datasets/TS021/editions/2021/versions/3).

In [37]:
ethnicity_total_field = "Ethnic group: Total: All usual residents"
# Select the leaf categories, excluding both the total and broad subtotals.
ethnicity_detail_fields = [
    field for field in ethnicity_support
    if field.startswith("Ethnic group:") and field.count(":") == 2 and field != ethnicity_total_field
]
assert len(ethnicity_detail_fields) == 19
ethnicity_feature_names = [
    "asian_bangladeshi_pct", "asian_chinese_pct", "asian_indian_pct",
    "asian_pakistani_pct", "asian_other_pct", "black_african_pct",
    "black_caribbean_pct", "black_other_pct", "mixed_white_asian_pct",
    "mixed_white_black_african_pct", "mixed_white_black_caribbean_pct",
    "mixed_other_pct", "white_british_pct", "white_irish_pct",
    "white_gypsy_irish_traveller_pct", "white_roma_pct", "white_other_pct",
    "other_arab_pct", "other_ethnic_group_pct",
]
ethnicity_counts = ethnicity_support[[ethnicity_total_field, *ethnicity_detail_fields]]
assert ethnicity_counts.notna().all().all()
assert ethnicity_counts.ge(0).all().all()
assert ethnicity_counts[ethnicity_total_field].gt(0).all()
ethnicity_display = pd.DataFrame({
    "area_code": ethnicity_support["geography code"],
    "local_area_name": ethnicity_support["geography code"].map(local_name_by_code),
})
for source_field, feature_name in zip(ethnicity_detail_fields, ethnicity_feature_names, strict=True):
    assert ethnicity_counts[source_field].le(ethnicity_counts[ethnicity_total_field]).all()
    ethnicity_display[feature_name] = 100 * ethnicity_counts[source_field] / ethnicity_counts[ethnicity_total_field]
assert ethnicity_display["area_code"].is_unique
assert ethnicity_display["local_area_name"].notna().all()
assert ethnicity_display[ethnicity_feature_names].ge(0).all().all()
assert ethnicity_display[ethnicity_feature_names].le(100).all().all()
# Keep a dictionary so concise CSV column names remain traceable to ONS categories.
ethnicity_dictionary = pd.DataFrame({
    "feature": ethnicity_feature_names,
    "source_field": ethnicity_detail_fields,
    "unit": "percent of all usual residents",
})
display(ethnicity_dictionary)
print(f"Compact table: {len(ethnicity_display):,} areas; {len(ethnicity_display.columns)} fields")
ethnicity_example = ethnicity_display.loc[ethnicity_display["area_code"].isin(birmingham_ethnicity["geography code"])].head(5)
ethnicity_preview = ethnicity_example.set_index("local_area_name").T
ethnicity_preview.columns.name = "Local area name →"
# Shade whole area columns, including their headers, to guide the eye downwards.
from IPython.display import HTML
preview_css = """<style>
.ethnicity-column-preview th, .ethnicity-column-preview td {
    color: #17212b !important; padding: 6px 12px;
    border-bottom: 1px solid #bdc8d1; text-align: right;
}
.ethnicity-column-preview th { background: #f2f4f7 !important; text-align: left; }
.ethnicity-column-preview tr > :nth-child(2n+2) { background: #dceefa !important; }
.ethnicity-column-preview tr > :nth-child(2n+3) { background: #fff0d2 !important; }
.ethnicity-column-preview thead th { white-space: nowrap; }
</style>"""
preview_html = ethnicity_preview.to_html(
    classes="ethnicity-column-preview", float_format=lambda value: f"{value:.2f}", border=0
)
display(HTML(preview_css + preview_html))
ethnicity_sum_difference = ethnicity_counts[ethnicity_detail_fields].sum(axis=1) - ethnicity_counts[ethnicity_total_field]
print(f"Largest absolute difference between detailed counts and published total: {ethnicity_sum_difference.abs().max():,} residents")
print("Preview rounded only; saved percentages retain precision.")

,feature,source_field,unit
0,asian_bangladeshi_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
1,asian_chinese_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
2,asian_indian_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
3,asian_pakistani_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
4,asian_other_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
5,black_african_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
6,black_caribbean_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
7,black_other_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
8,mixed_white_asian_pct,Ethnic group: Mixed or Multiple ethnic groups:...,percent of all usual residents
9,mixed_white_black_african_pct,Ethnic group: Mixed or Multiple ethnic groups:...,percent of all usual residents


Compact table: 660 areas; 21 fields


Local area name →,Hill Hook,Four Oaks,Little Sutton,Sutton Coldfield North,Reddicap Heath
area_code,E02001827,E02001828,E02001829,E02001830,E02001831
asian_bangladeshi_pct,0.37,0.28,0.41,0.50,0.40
asian_chinese_pct,1.21,1.12,1.55,1.05,0.81
asian_indian_pct,8.02,5.03,7.89,9.57,3.52
asian_pakistani_pct,1.95,1.76,1.93,2.27,1.78
asian_other_pct,1.63,0.95,0.87,0.79,1.15
black_african_pct,0.78,0.66,0.68,0.71,1.59
black_caribbean_pct,0.99,0.99,0.88,0.84,2.34
black_other_pct,0.31,0.22,0.23,0.35,0.70
mixed_white_asian_pct,1.35,0.98,0.75,1.01,1.04


Largest absolute difference between detailed counts and published total: 0 residents
Preview rounded only; saved percentages retain precision.


### Save ethnicity

The 21-column table goes in `Data/Cleaned/Ethnicity/`, with the 28-field counts and a column dictionary alongside. Backed up on rerun, then read back and checked.

In [38]:
ethnicity_folder = project_data("Cleaned/Ethnicity")
ethnicity_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("ethnicity_2021_msoa.csv", ethnicity_display),
    ("ethnicity_2021_msoa_support.csv", ethnicity_support),
    ("ethnicity_2021_feature_dictionary.csv", ethnicity_dictionary),
]:
    destination = ethnicity_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Ethnicity"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified ethnicity_2021_msoa.csv: 660 rows; 21 fields
Saved and verified ethnicity_2021_msoa_support.csv: 660 rows; 28 fields
Saved and verified ethnicity_2021_feature_dictionary.csv: 19 rows; 3 fields


The mix varies a lot. White British residents range from 2.7% of an area (Sparkhill North) to 97.1%, with a median of 80.3%, and 123 of my 660 areas are under 50% White British. Pakistani residents reach 61.8% in one area.

These 19 percentages are shares of the same population, so they're closely related. I'll need to allow for that when I choose model inputs.

## 10. Recorded crime

Recorded crime gives me context about the local environment. The source is January to December 2024 street-level crime and anti-social behaviour records from seven police forces, including British Transport Police. Locations are anonymised. The 2021 LSOA codes describe the geography and not the year of the incidents.

I use the LSOA-to-MSOA lookup to pick my 660 areas, look at the fields and categories, and calculate rates. 2024 is later than the HLE period, so this is later context and can't be treated as evidence from the same years. Source notes: [Police open data](https://data.police.uk/about/) and the [changelog](https://data.police.uk/changelog/).

In [39]:
crime_zip = raw_file("police_recorded_crime_2024_regional_forces.zip")
with ZipFile(crime_zip) as archive:
    crime_members = sorted(name for name in archive.namelist() if name.endswith("-street.csv"))
    crime_frames = []
    for name in crime_members:
        with archive.open(name) as source_file:
            frame = pd.read_csv(source_file, dtype=str)
        frame["source_file"] = name
        crime_frames.append(frame)
crime_raw = pd.concat(crime_frames, ignore_index=True)
expected_months = {f"2024-{month:02d}" for month in range(1, 13)}
assert len(crime_members) == 84
assert set(crime_raw["Month"]) == expected_months
crime_lookup = pd.read_csv(project_data("Prepared/Geography/ons_lsoa21_to_msoa21_lad22_exploratory_west_midlands.csv"))
assert crime_lookup["LSOA21CD"].is_unique
crime_lookup = crime_lookup.loc[crime_lookup["MSOA21CD"].isin(coverage_codes)]
crime_selected = crime_raw.merge(
    crime_lookup[["LSOA21CD", "MSOA21CD", "LAD22NM"]],
    left_on="LSOA code", right_on="LSOA21CD", how="inner", validate="many_to_one",
)
assert set(crime_selected["MSOA21CD"]) == coverage_codes
print(f"Regional download: {len(crime_raw):,} rows; 12 original fields; {len(crime_members)} monthly force CSVs")
print(f"Selected geography: {len(crime_selected):,} rows; {crime_selected['MSOA21CD'].nunique():,} MSOAs")
print(f"Download rows without an LSOA code: {crime_raw['LSOA code'].isna().sum():,}; not assignable to an MSOA")

Regional download: 846,166 rows; 12 original fields; 84 monthly force CSVs
Selected geography: 539,609 rows; 660 MSOAs
Download rows without an LSOA code: 7,786; not assignable to an MSOA


### Fields and categories

The original 12 fields and a Birmingham sample, then category totals for my areas. I add source-file and lookup fields for traceability. They aren't crime measures. Anti-social behaviour stays separate from recorded crime. The publisher combines violence and sexual offences into one category, so I can't split them.

In [40]:
crime_original_fields = [field for field in crime_raw if field != "source_file"]
crime_field_notes = [
    "Published identifier; may be missing or repeated", "Incident month",
    "Reporting police force", "Published force attribution",
    "Anonymised longitude", "Anonymised latitude", "Anonymised location description",
    "2021 LSOA code — geographical join key", "Published LSOA name",
    "Recorded category, including ASB", "Latest published outcome", "Additional context, often empty",
]
display(pd.DataFrame({"Source field": crime_original_fields, "What it contains": crime_field_notes}))
birmingham_crime = crime_selected.loc[crime_selected["LAD22NM"].eq("Birmingham")]
with pd.option_context("display.max_columns", None):
    display(birmingham_crime[crime_original_fields].head(5).reset_index(drop=True))
crime_category_counts = crime_selected["Crime type"].value_counts().rename_axis("category").reset_index(name="published_rows")
display(crime_category_counts)

,Source field,What it contains
0,Crime ID,Published identifier; may be missing or repeated
1,Month,Incident month
2,Reported by,Reporting police force
3,Falls within,Published force attribution
4,Longitude,Anonymised longitude
5,Latitude,Anonymised latitude
6,Location,Anonymised location description
7,LSOA code,2021 LSOA code — geographical join key
8,LSOA name,Published LSOA name
9,Crime type,"Recorded category, including ASB"


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2024-01,British Transport Police,British Transport Police,-1.828638,52.578809,On or near Four Oaks (Station),E01009427,Birmingham 004C,Vehicle crime,NaN,NaN
1,NaN,2024-01,British Transport Police,British Transport Police,-1.839273,52.528400,On or near Erdington (Station),E01009007,Birmingham 023D,Other theft,NaN,NaN
2,NaN,2024-01,British Transport Police,British Transport Police,-1.901709,52.516396,On or near Perry Barr (Station),E01009242,Birmingham 024E,Public order,NaN,NaN
3,NaN,2024-01,British Transport Police,British Transport Police,-1.913330,52.489921,On or near Jewellery Quarter (Station),E01033559,Birmingham 049E,Public order,NaN,NaN
4,NaN,2024-01,British Transport Police,British Transport Police,-1.892120,52.478692,On or near Birmingham Moor Street (Station),E01033561,Birmingham 050E,Criminal damage and arson,NaN,NaN


,category,published_rows
0,Violence and sexual offences,209057
1,Anti-social behaviour,55605
2,Shoplifting,45934
3,Vehicle crime,44158
4,Criminal damage and arson,40241
5,Other theft,35727
6,Public order,34963
7,Burglary,25330
8,Drugs,12513
9,Other crime,11543


### Coverage and repeated IDs

Every one of my MSOAs has records in all 12 months, and the table shows monthly totals by force. That shows records exist. It doesn't prove every incident was reported or recorded. The changelog lists past reporting gaps, and my files now have data for every force in every month.

Some crime IDs repeat, some with different locations. I count published rows and keep a review table. I don't pick a location arbitrarily or quietly drop duplicates, because even identical rows can be ambiguous here. So the rates stay provisional until I've reviewed the repeated IDs. Records with no location stay out, since I can't assign them to an area and can't read them as zero crime.

In [41]:
crime_force_month = crime_raw.groupby(["Reported by", "Month"]).size().unstack(fill_value=0)
assert crime_force_month.shape == (7, 12)
assert crime_force_month.gt(0).all().all()
display(crime_force_month)
crime_months = crime_selected.groupby("MSOA21CD")["Month"].nunique()
assert crime_months.eq(12).all()
identified_crime = crime_selected.loc[crime_selected["Crime ID"].notna()]
crime_repeated_id_review = identified_crime.loc[
    identified_crime.duplicated(["Month", "Crime ID"], keep=False)
].copy()
crime_repeated_counts = crime_repeated_id_review.groupby("MSOA21CD").size()
print(f"MSOAs with records in all 12 months: {crime_months.eq(12).sum():,}")
print(f"Selected rows with a repeated month/ID: {len(crime_repeated_id_review):,}")
print(f"MSOAs containing these review rows: {crime_repeated_id_review['MSOA21CD'].nunique():,}")
print(f"Exact repeated original-field rows with non-missing ID: {identified_crime.duplicated(crime_original_fields).sum():,}")

Month,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
Reported by,,,,,,,,,,,,
British Transport Police,3376,3319,3481,3301,3519,3425,3526,3599,3311,3629,3545,3309
Derbyshire Constabulary,8216,8902,8910,8809,9491,9646,9543,9223,8724,9066,6964,7742
Leicestershire Police,8324,8243,8906,9128,9394,9490,9563,9357,9155,9172,8953,8216
Staffordshire Police,8504,7583,7888,9010,8857,7976,9243,8596,8502,8518,7696,7695
Warwickshire Police,3882,3780,3966,4209,4447,4221,4586,4217,4092,4260,3716,3684
West Mercia Police,7700,7654,8529,8540,9011,8917,8382,9063,8207,8500,8135,7757
West Midlands Police,27500,26067,28383,28495,31037,29865,31636,29180,28635,29074,26571,25723


MSOAs with records in all 12 months: 660
Selected rows with a repeated month/ID: 1,224
MSOAs containing these review rows: 65


Exact repeated original-field rows with non-missing ID: 0


### Two rates

I keep two rates per 1,000 residents: published recorded-crime rows (without anti-social behaviour) and published anti-social behaviour rows, kept separate. Rates stop areas of different sizes being compared on raw totals. They count incidents. They aren't the percentage of residents who experienced crime.

The denominator is the Census 2021 usual-resident population from TS007A. It's a fixed base and not a 2024 estimate, so population change since 2021 affects the rates. A city centre, shopping area or transport hub also has visitors who aren't residents, so a high rate there reflects footfall as well as local residents. This is recorded activity against resident population. It isn't anyone's personal risk.

In [42]:
with ZipFile(raw_file("census2021-ts007a.zip")) as archive:
    with archive.open("census2021-ts007a-msoa.csv") as source_file:
        crime_population_raw = pd.read_csv(source_file)
assert crime_population_raw["geography code"].is_unique
assert crime_population_raw["date"].astype(str).eq("2021").all()
crime_population = crime_population_raw.set_index("geography code")["Age: Total"]
crime_area_counts = pd.crosstab(crime_selected["MSOA21CD"], crime_selected["Crime type"]).reindex(sorted(coverage_codes), fill_value=0)
crime_support = crime_area_counts.copy()
crime_support.index.name = "area_code"
crime_support = crime_support.reset_index()
crime_support.insert(1, "local_area_name", crime_support["area_code"].map(local_name_by_code))
crime_support["population_2021"] = crime_support["area_code"].map(crime_population)
assert crime_support["population_2021"].notna().all() and crime_support["population_2021"].gt(0).all()
crime_categories = [category for category in crime_area_counts.columns if category != "Anti-social behaviour"]
crime_support["recorded_crime_rows"] = crime_support[crime_categories].sum(axis=1)
crime_support["months_with_records"] = crime_support["area_code"].map(crime_months)
crime_support["repeated_id_review_rows"] = crime_support["area_code"].map(crime_repeated_counts).fillna(0).astype(int)
crime_display = crime_support[["area_code", "local_area_name"]].copy()
crime_display["recorded_crime_rows_per_1000_residents"] = 1000 * crime_support["recorded_crime_rows"] / crime_support["population_2021"]
crime_display["asb_rows_per_1000_residents"] = 1000 * crime_support["Anti-social behaviour"] / crime_support["population_2021"]
assert crime_display["area_code"].is_unique and crime_display["local_area_name"].notna().all()
assert crime_display.iloc[:, 2:].notna().all().all() and crime_display.iloc[:, 2:].ge(0).all().all()
assert crime_support[crime_area_counts.columns].sum().sum() == len(crime_selected)
print(f"Compact table: {len(crime_display):,} MSOAs; {len(crime_display.columns)} fields")
birmingham_crime_codes = set(birmingham_crime["MSOA21CD"])
display(crime_display.loc[crime_display["area_code"].isin(birmingham_crime_codes)].head(10).round(2))
print("Preview rounded only; exported rates retain precision. Repeated-ID caveat remains unresolved.")

Compact table: 660 MSOAs; 4 fields


Crime type,area_code,local_area_name,recorded_crime_rows_per_1000_residents,asb_rows_per_1000_residents
0,E02001827,Hill Hook,34.41,3.41
1,E02001828,Four Oaks,78.85,3.64
2,E02001829,Little Sutton,31.77,1.86
3,E02001830,Sutton Coldfield North,49.15,4.85
4,E02001831,Reddicap Heath,78.05,8.98
5,E02001832,Rectory Park,90.91,6.43
6,E02001833,Kingstanding North,61.32,6.40
7,E02001834,Old Oscott,80.48,4.91
8,E02001835,Sutton Coldfield South,118.39,18.06
9,E02001836,New Oscott,99.60,9.01


Preview rounded only; exported rates retain precision. Repeated-ID caveat remains unresolved.


### Save crime

The four-column table goes in `Data/Cleaned/Crime/`, with category counts and denominators, force-month coverage and the repeated-ID rows in separate files. The raw ZIP stays intact. Backed up on rerun, then read back and checked.

In [43]:
crime_folder = project_data("Cleaned/Crime")
crime_folder.mkdir(parents=True, exist_ok=True)
crime_coverage_export = crime_force_month.reset_index()
for filename, table in [
    ("crime_2024_msoa_provisional.csv", crime_display),
    ("crime_2024_msoa_support.csv", crime_support),
    ("crime_2024_force_month_coverage.csv", crime_coverage_export),
    ("crime_2024_repeated_id_review.csv", crime_repeated_id_review),
]:
    destination = crime_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Crime"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    # Crime identifier fields are text; preserve them explicitly when reading back.
    reloaded = pd.read_csv(destination, dtype=table.dtypes.to_dict())
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True), check_names=False)
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified crime_2024_msoa_provisional.csv: 660 rows; 4 fields
Saved and verified crime_2024_msoa_support.csv: 660 rows; 20 fields
Saved and verified crime_2024_force_month_coverage.csv: 7 rows; 13 fields
Saved and verified crime_2024_repeated_id_review.csv: 1,224 rows; 16 fields


In [ ]:
crime_support = pd.read_csv(project_data("Cleaned/Crime/crime_2024_msoa_support.csv"))
published_rows = crime_support["recorded_crime_rows"] + crime_support["Anti-social behaviour"]
repeated_rows = crime_support["repeated_id_review_rows"]
print(f"Rows sharing an ID with another row: {repeated_rows.sum():,} of {published_rows.sum():,} published rows ({100 * repeated_rows.sum() / published_rows.sum():.2f}%)")
share_repeated = 100 * repeated_rows / published_rows
print(f"Areas affected: {(repeated_rows > 0).sum()}; biggest share in one area: {share_repeated.max():.1f}% ({crime_support.loc[share_repeated.idxmax(), 'local_area_name']})")

# Worst case: drop every repeated row, then see whether the order of the areas changes.
rate_published = 1000 * published_rows / crime_support["population_2021"]
rate_worst_case = 1000 * (published_rows - repeated_rows) / crime_support["population_2021"]
print("Rank correlation, published rows against the worst case:", round(rate_published.rank().corr(rate_worst_case.rank()), 4))

Rows sharing an ID with another row: 1,224 of 539,609 published rows (0.23%)
Areas affected: 65; biggest share in one area: 7.5% (Swadlincote West)
Rank correlation, published rows against the worst case: 0.9998


Of 846,166 records downloaded, 539,609 fall in my 660 areas, and 7,786 have no location so I can't place them. Recorded crime runs from 19.8 per 1,000 residents (Monkspath North) to 1,085 (Birmingham city centre), against a median of 76.5. The city centre is about 14 times the median. That's visitors and footfall, and section 11 deals with it.

1,224 rows share a crime ID with another row and sit in a review file. The check above shows they're only 0.2% of the published rows. Even in the worst case, where I drop every one of them, the order of the areas hardly changes. So I keep the published rows.

## 11. Areas that need care

Some areas are very different from the rest. So I flag them. Nothing is deleted, every area stays in every measure except one: crime. The only thing I set aside is the crime figure, especially for visitor hubs.

**Visitor hubs.** Recorded crime in this project is counted against residents, but city and town centres have far more people passing through than live there. So, Birmingham city centre is 14 times the median rate, and it isn't the only one. So I flag areas above the usual outlier fence for the data: my AI robot told me the third quartile plus three times the inter-quartile range was a fair filter. But a simpler rule, more than four times the median, picks the same areas. So these outlier areas are left out of the crime analysis only. Their deprivation, pollution, income, qualifications and everything else stay in.

**Student areas.** I know my city, and I know that in some areas most residents are students, so the employment and qualification shares describe students and not the local economy. I work out full-time students as a share of residents aged 16+. Most areas sit around the median. A handful sit far above it, and there's a clear break between them and the rest. I draw the line at 50%. These areas are flagged and kept in everything. The flag is a caveat for reading their results.

**Birmingham city centre.** It gets a caveat of its own. It's not like anywhere else: it has the highest NO2 and the highest share of people with degrees in my data, and its crime rate is far above any other area. Any result that leans on this location is skewed and so needs reading with that in mind.

Both rules are mine and not ONS's, so I check the list by eye with the aid of my AI coding robot Kevin. It spots the outliers, and I check that it hasn't missed any. The student-area rule is clear-cut, but the visitor-hub rule is a bit more subjective. I check the flagged areas against my knowledge of the city and its suburbs, and I add or remove a few. The final list is in `Data/Reference/Area Flags/area-flags.csv`, with a note on how each area was flagged.


In [ ]:
import shutil
from datetime import datetime

# Full-time students as a share of residents aged 16+, from the saved economic activity counts.
activity_counts = pd.read_csv(project_data("Cleaned/Economic Activity/economic_activity_2021_msoa_support.csv"))
residents_16plus = "Economic activity status: Total: All usual residents aged 16 years and over"
student_counts = [
    "Economic activity status: Economically inactive: Student",
    "Economic activity status: Economically active and a full-time student",
]
students = pd.DataFrame({
    "area_code": activity_counts["geography code"],
    "full_time_student_share_pct": 100 * activity_counts[student_counts].sum(axis=1) / activity_counts[residents_16plus],
})

crime_rates = pd.read_csv(project_data("Cleaned/Crime/crime_2024_msoa_provisional.csv"))
flags = crime_rates[["area_code", "local_area_name", "recorded_crime_rows_per_1000_residents"]].merge(
    students, on="area_code", validate="one_to_one"
)
assert len(flags) == 660

student_limit = 50                       # % of residents aged 16+
crime_rate = flags["recorded_crime_rows_per_1000_residents"]
q1, q3 = crime_rate.quantile([0.25, 0.75])
crime_limit = q3 + 3 * (q3 - q1)         # the usual outlier fence

flags["student_dominated"] = flags["full_time_student_share_pct"] > student_limit
flags["visitor_hub"] = crime_rate > crime_limit
flags["exclude_from_crime_analysis"] = flags["visitor_hub"]       # crime only; nothing else is excluded
flags["city_centre_caveat"] = flags["local_area_name"].str.contains("Birmingham city centre", case=False)
assert flags["city_centre_caveat"].sum() == 1
flagged = flags["student_dominated"] | flags["visitor_hub"]
flags["reason"] = [
    " + ".join(label for label, applies in [("student area", s), ("visitor hub", h)] if applies)
    for s, h in zip(flags["student_dominated"], flags["visitor_hub"])
]

below_limit = flags.loc[~flags["student_dominated"], "full_time_student_share_pct"]
print(f"Student share: median {flags['full_time_student_share_pct'].median():.1f}%; "
      f"above {student_limit}%: {flags['student_dominated'].sum()} areas; highest below the line: {below_limit.max():.1f}%")
print(f"Crime fence: {crime_limit:.0f} per 1,000; above it: {flags['visitor_hub'].sum()} areas")
print("Same areas with the 'more than 4 times the median' rule:",
      set(flags.loc[crime_rate > 4 * crime_rate.median(), "area_code"]) == set(flags.loc[flags["visitor_hub"], "area_code"]))
display(flags.loc[flagged].sort_values("full_time_student_share_pct", ascending=False)
        [["local_area_name", "full_time_student_share_pct", "recorded_crime_rows_per_1000_residents", "reason", "exclude_from_crime_analysis"]].round(1).reset_index(drop=True))

flags_folder = project_data("Cleaned/Modelling Flags")
flags_folder.mkdir(parents=True, exist_ok=True)
flags_file = flags_folder / "modelling_flags_msoa.csv"
if flags_file.exists():
    backup_folder = project_folder / "VS_CODE/Backups/Outputs/Modelling Flags"
    backup_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy2(flags_file, backup_folder / f"{flags_file.stem} - {datetime.now():%Y%m%d-%H%M%S-%f}{flags_file.suffix}")
saved_flags = flags.round({"full_time_student_share_pct": 2, "recorded_crime_rows_per_1000_residents": 2})
saved_flags.to_csv(flags_file, index=False)
check = pd.read_csv(flags_file)
assert len(check) == 660 and list(check.columns) == list(saved_flags.columns)
assert int(check["exclude_from_crime_analysis"].sum()) == int(flags["visitor_hub"].sum())
print(f"Saved and verified {flags_file.name}: {len(check)} rows; {int(flagged.sum())} areas flagged; "
      f"{int(check['exclude_from_crime_analysis'].sum())} set aside for crime only")

Student share: median 6.1%; above 50%: 6 areas; highest below the line: 34.4%
Crime fence: 281 per 1,000; above it: 6 areas
Same areas with the 'more than 4 times the median' rule: True


,local_area_name,full_time_student_share_pct,recorded_crime_rows_per_1000_residents,reason,exclude_from_crime_analysis
0,Selly Oak,81.7,113.5,student area,False
1,North Central,61.1,316.4,student area + visitor hub,True
2,Central Coventry,58.2,433.3,student area + visitor hub,True
3,Lower Stoke,52.6,200.0,student area,False
4,Cannon Park,52.4,65.1,student area,False
5,Edgbaston South,51.2,79.6,student area,False
6,Digbeth,22.2,615.1,visitor hub,True
7,Birmingham city centre,19.7,1085.4,visitor hub,True
8,Wolverhampton Central,19.6,344.9,visitor hub,True
9,Walsall Central,13.9,351.8,visitor hub,True


Saved and verified modelling_flags_msoa.csv: 660 rows; 10 areas flagged; 6 set aside for crime only


Ten areas are flagged: six student areas and six visitor hubs, with Central Coventry and North Central in both. The student areas are Selly Oak, North Central, Central Coventry, Lower Stoke, Cannon Park and Edgbaston South. The visitor hubs are Birmingham city centre, Digbeth, Central Coventry, Walsall Central, Wolverhampton Central and North Central. That's 10 of 660 areas.

Only the six visitor hubs are left out, and only of the crime analysis. The student areas stay in everything, with the flag as a caveat. Birmingham city centre carries its own caveat in `city_centre_caveat`.

Areas with a smaller student population stay in. Hillfields is the highest of those at 34%. I keep every area's student share in `modelling_flags_msoa.csv`, so I can test later whether it matters. The same file says which areas are flagged and why.


## 12. What I now have

Eleven prepared datasets, all on the same 660 areas and keyed on the MSOA code. The table gives the source, the period and the main catch for each.

| Dataset | Source | Period | Original grain | Main catch |
|---|---|---|---|---|
| HLE | ONS health state life expectancy | 2019–2023 | MSOA, by sex | ONS imputed some inputs. Keep the confidence intervals. |
| Household deprivation | Census 2021 TS011 | 2021 | MSOA | The health dimension overlaps with HLE. |
| Overcrowding | Census 2021 TS052 | 2021 | MSOA | Bedroom shortfall only. |
| Fuel poverty | Fuel poverty statistics, Table 4 | 2024 | LSOA, combined to MSOA | After the HLE period. Small-area estimates. |
| NO₂ and PM2.5 | Defra modelled background | 2024 | 1 km grid | After the HLE period. Area-weighted, not population-weighted. |
| Economic activity | Census 2021 TS066 | 2021 | MSOA | Student areas distort the employment share (section 11). |
| Qualifications | Census 2021 TS067 | 2021 | MSOA | Three measures only. Student areas look odd (section 11). |
| Household income | ONS small area income estimates | Year ending March 2023 | MSOA | Modelled averages. |
| Ethnicity | Census 2021 TS021 | 2021 | MSOA | 19 related shares. |
| Recorded crime | Police open data | 2024 | LSOA, combined to MSOA | City and town centres inflate the rate (section 11). After the HLE period. |

The code below checks every saved compact file against my 660 boundary areas and shows the range of each measure.

In [ ]:
prepared_files = {
    "HLE": "HLE/hle_2019_2023_msoa.csv",
    "Household deprivation": "Household Deprivation/household_deprivation_2021_msoa.csv",
    "Overcrowding": "Overcrowding/overcrowding_2021_msoa.csv",
    "Fuel poverty": "Fuel Poverty/fuel_poverty_2024_msoa.csv",
    "NO2": "Pollution/no2_2024_msoa.csv",
    "PM2.5": "Pollution/pm25_2024_msoa.csv",
    "Economic activity": "Economic Activity/economic_activity_2021_msoa.csv",
    "Qualifications": "Qualifications/qualifications_2021_msoa.csv",
    "Income": "Income/income_after_housing_costs_fye2023_msoa.csv",
    "Ethnicity": "Ethnicity/ethnicity_2021_msoa.csv",
    "Crime": "Crime/crime_2024_msoa_provisional.csv",
}
tables = {name: pd.read_csv(project_data("Cleaned/" + relative)) for name, relative in prepared_files.items()}

checks = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(table),
        "Areas": table["area_code"].nunique(),
        "Same 660 areas as the boundary file": set(table["area_code"]) == coverage_codes,
        "Missing cells": int(table.isna().sum().sum()),
    }
    for name, table in tables.items()
])
display(checks)
assert checks["Same 660 areas as the boundary file"].all() and checks["Missing cells"].eq(0).all()

# Range of every measure. Ethnicity has 19 shares, so it's left out here.
range_rows = []
for name, table in tables.items():
    if name == "Ethnicity":
        continue
    parts = [(f"HLE, {sex.lower()}", part.reset_index(drop=True)) for sex, part in table.groupby("sex")] if name == "HLE" else [(None, table)]
    for label, part in parts:
        for measure in [c for c in part.columns if c not in ("area_code", "local_area_name", "sex")]:
            low, high = part[measure].idxmin(), part[measure].idxmax()
            range_rows.append({
                "Measure": label or measure,
                "Lowest": round(part.loc[low, measure], 2), "Where (lowest)": part.loc[low, "local_area_name"],
                "Median": round(part[measure].median(), 2),
                "Highest": round(part.loc[high, measure], 2), "Where (highest)": part.loc[high, "local_area_name"],
            })
display(pd.DataFrame(range_rows))

,Dataset,Rows,Areas,Same 660 areas as the boundary file,Missing cells
0,HLE,1320,660,True,0
1,Household deprivation,660,660,True,0
2,Overcrowding,660,660,True,0
3,Fuel poverty,660,660,True,0
4,NO2,660,660,True,0
5,PM2.5,660,660,True,0
6,Economic activity,660,660,True,0
7,Qualifications,660,660,True,0
8,Income,660,660,True,0
9,Ethnicity,660,660,True,0


,Measure,Lowest,Where (lowest),Median,Highest,Where (highest)
0,"HLE, female",52.90,Sparkbrook South,64.85,76.60,Kingswood
1,"HLE, male",52.40,Bloxwich South,63.50,75.40,Monkspath North
2,households_deprived_2plus_pct,6.06,Monkspath North,19.26,41.46,Sparkbrook South
3,overcrowded_households_pct,0.52,Weston,2.85,29.01,Saltley East
4,fuel_poor_households_pct,4.03,Monkspath North,11.08,28.20,Lower Stoke
5,no2_area_mean_ug_m3,2.50,Wigmore,10.26,21.63,Birmingham city centre
6,pm25_area_mean_ug_m3,4.76,Wigmore,7.26,8.70,Sparkhill North
7,employed_residents_16plus_pct,21.48,Selly Oak,55.83,71.48,Hawks Green
8,unemployment_rate_active_pct,1.78,Loggerheads,5.50,29.75,North Central
9,no_qualifications_pct,5.36,Selly Oak,20.90,38.28,Friar Park


All eleven files hold the same 660 areas with no missing cells. HLE has two rows per area, one for each sex, and the others have one row per area. A twelfth file, `modelling_flags_msoa.csv`, lists the flagged areas and why.

Three things carry forward:

- **Ten areas are flagged** (section 11). Six visitor hubs are left out of the crime analysis only. Student areas stay in everything, with a caveat. Birmingham city centre has a caveat of its own. All rows stay in the files, and the flags file says which areas and why.
- **Crime uses the published rows.** The repeated IDs make almost no difference.
- **The later periods.** Fuel poverty, pollution and crime are from 2024, after the HLE period. Fuel poverty in England was stable from 2019 and has fallen since 2022 ([government fuel poverty factsheet, 2025](https://assets.publishing.service.gov.uk/media/67e3d2ebd4a1b0665b8ee2c3/Factsheet_2025.pdf)), so I don't expect the later year to reshuffle the areas much. I'm assuming the same holds for pollution and crime. I'll state that as a limit and I won't adjust for it.

The next step is to join these to HLE by area code and see how they line up.